# Consumer Complaint Classification - Training and evaluation

Upload this notebook to Kaggle, enable Internet and GPU T4 x2, then run from the top. The embedded implementation downloads the pinned dataset, verifies its reference fingerprints, prepares shared splits, trains four models, selects on validation, evaluates the held-out test set and exports the selected classifier.

**Saved outputs document the completed historical experiment.** Application and inference sources were subsequently corrected and synchronized with the local project on 10 November 2026. These source corrections were tested locally; full GPU training has not been rerun. A new run has a new code signature, so historical checkpoint reuse requires the original code/environment.

Source text was already processed by its publisher. This notebook embeds neither the full corpus nor pretrained weights nor credentials. If dependency repair requests a kernel restart, restart once while retaining the installed environment and then run from the top.


In [1]:
# import os
# os._exit(0)

In [2]:
# Environment setup — run in a fresh Kaggle kernel before importing Transformers.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ.setdefault("KAGGLEHUB_CACHE", "/kaggle/working/kagglehub_cache")
import importlib.metadata as metadata
from pathlib import Path
import subprocess
import sys

PINNED = {"transformers": "4.57.1", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2", "safetensors": "0.8.0", "numpy": "2.3.5", "pandas": "2.2.3", "scikit-learn": "1.8.0"}
def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def ensure_pinned_packages(pinned, get_version, loaded_modules, install, restart_pending=False):
    """Install first; a fresh kernel is needed only for changed loaded modules.

    Arguments are injectable so this policy can be tested without pip/imports.
    """
    module_names = {"scikit-learn": "sklearn", "huggingface-hub": "huggingface_hub"}
    changed = [name for name, version in pinned.items() if get_version(name) != version]
    changed_loaded = [name for name in changed if module_names.get(name, name) in loaded_modules]
    needed = [f"{name}=={pinned[name]}" for name in changed]
    for name in ["matplotlib", "tqdm", "kagglehub"]:
        if get_version(name) is None:
            needed.append(name)
    if needed:
        print("Installing required dependencies before checking restart requirements...", flush=True)
        install(needed)
    missing = {name: get_version(name) for name, version in pinned.items() if get_version(name) != version}
    if missing:
        raise RuntimeError(f"Dependency installation did not provide the required versions: {missing}")
    # Detect an old in-memory import even if pip already changed its disk version.
    stale_loaded = []
    for name, version in pinned.items():
        module = loaded_modules.get(module_names.get(name, name))
        loaded_version = getattr(module, "__version__", None) if module is not None else None
        if loaded_version is not None and str(loaded_version) != version:
            stale_loaded.append(name)
    return bool(restart_pending or changed_loaded or stale_loaded)


if os.environ.get("COMPLAINTS_SKIP_INSTALL") != "1":
    _COMPLAINTS_RESTART_REQUIRED = ensure_pinned_packages(
        PINNED, installed_version, sys.modules,
        lambda requirements: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements]),
        globals().get("_COMPLAINTS_RESTART_REQUIRED", False),
    )
    if _COMPLAINTS_RESTART_REQUIRED:
        raise RuntimeError(
            "Required package versions are now installed successfully. Kaggle had already loaded older packages. "
            "Restart the kernel/session ONCE, keeping this same notebook and environment (do not factory-reset the runtime), "
            "then Run All from the top. Re-running this cell without restarting cannot unload those packages. "
            "After a normal kernel restart the installed versions are reused."
        )
    # Text-only Transformers does not need vision/audio libraries. Remove an
    # optional preinstalled package only if it is installed but cannot import.
    for optional in ["torchvision", "librosa"]:
        if installed_version(optional) is not None:
            probe = subprocess.run([sys.executable, "-c", f"import {optional}"], capture_output=True, text=True)
            if probe.returncode:
                print(f"Removing broken optional {optional}; it is unused by this text-classification project.")
                subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", optional])

import torch
from packaging.version import Version
if sys.version_info < (3, 11):
    raise RuntimeError("Use a Kaggle Python 3.11+ image.")
if Version(torch.__version__.split("+")[0]) < Version("2.3"):
    raise RuntimeError("This notebook requires torch >= 2.3. Select a current Kaggle image; keep its CUDA-enabled torch build.")
from transformers import AutoModelForSequenceClassification, AutoTokenizer
assert callable(AutoModelForSequenceClassification.from_pretrained) and callable(AutoTokenizer.from_pretrained)
PROJECT_DIR = Path(os.environ.get("COMPLAINTS_CODE_DIR", "/kaggle/working/consumer_complaints_code"))
(PROJECT_DIR / "src/complaints").mkdir(parents=True, exist_ok=True)
print("Environment ready:", torch.__version__, "Transformers", metadata.version("transformers"))


Environment ready: 2.11.0+cu130 Transformers 4.57.1


## Embedded runtime scripts

These cells write the complete implementation into Kaggle's working directory. They can be collapsed after execution. The source is included directly in this notebook so it remains portable.


In [3]:
# Embedded file: src/complaints/__init__.py
embedded_source = '"""Reproducible consumer complaint classification."""\n\n__version__ = "1.0.0"\n'
target = PROJECT_DIR / 'src/complaints/__init__.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/__init__.py


In [4]:
# Embedded file: src/complaints/bootstrap.py
embedded_source = '"""Download the pinned source and transactionally prepare/audit Kaggle inputs."""\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\nSOURCE = \'shashwatwork/consume-complaints-dataset-fo-nlp/versions/1\'\nCSV_SHA = \'a120ac8450cb3839bf0085bbae68f7c0f3b5351302cafd0bc85ae27a251010b4\'\n\n\ndef digest(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f,\'sha256\').hexdigest()\n\n\ndef canonical(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef verify_csv(path):\n    if digest(path)!=CSV_SHA:\n        raise RuntimeError(\'Downloaded CSV does not match the audited version 1 SHA-256. No alternative source will be used.\')\n    return Path(path)\n\n\ndef reference_check(root, reference):\n    root=Path(root)\n    differences={}\n    actual=read(root/\'audit.json\')\n    keys=[\'retained_rows\',\'split_counts\',\'class_counts_by_split\',\'labels\',\'exclusion_flow\',\n          \'source_rows\',\'source_columns\',\'duplicate_overlap_between_splits\',\'split_group_count\',\n          \'encoded_collision_grouping_passes\']\n    for key in keys:\n        if actual[key]!=reference[\'audit\'][key]:\n            differences[key]={\'actual\':actual[key],\'expected\':reference[\'audit\'][key]}\n    with gzip.open(root/\'records.csv.gz\',\'rb\') as f:\n        records_hash=hashlib.file_digest(f,\'sha256\').hexdigest()\n    if records_hash!=reference[\'records_csv_sha256\']:\n        differences[\'records_and_split_assignments\']=\'Decompressed canonical CSV hash differs\'\n    if canonical(read(root/\'vocabulary.json\'))!=reference[\'vocabulary_sha256\']:\n        differences[\'training_vocabulary\']=\'Canonical vocabulary hash differs\'\n    with zipfile.ZipFile(root/\'encoded_inputs.npz\') as z:\n        if set(z.namelist())!=set(reference[\'npy_sha256\']):\n            differences[\'encoded_array_names\']=\'Array members differ\'\n        for name,expected in reference[\'npy_sha256\'].items():\n            if name not in z.namelist() or hashlib.sha256(z.read(name)).hexdigest()!=expected:\n                differences[name]=\'Encoded array shape/dtype/content differs\'\n    report={\'status\':\'failed\' if differences else \'passed\',\'differences\':differences,\n            \'method\':\'Canonical record CSV, vocabulary, NPY array bytes and audited counts\',\n            \'reference_source_csv_sha256\':CSV_SHA}\n    (root/\'reference_comparison.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    if differences:\n        raise RuntimeError(f\'Preparation differs from supplied reference. Inspect {root / "reference_comparison.json"}; do not train until investigated.\')\n    return report\n\n\ndef preparation_identity(project_dir):\n    project_dir=Path(project_dir)\n    versions={name:importlib.metadata.version(name) for name in\n              [\'numpy\',\'pandas\',\'scikit-learn\',\'transformers\',\'tokenizers\']}\n    return {\'source_csv_sha256\':CSV_SHA,\n            \'config_sha256\':canonical(read(project_dir/\'configs/experiment.json\')),\n            \'reference_sha256\':digest(project_dir/\'configs/reference.json\'),\n            \'code\':{name:digest(project_dir/\'src/complaints\'/name) for name in\n                    [\'preparation.py\',\'bootstrap.py\',\'text.py\']},\'versions\':versions}\n\n\ndef prepare_all(project_dir, prepared_dir):\n    from .data import PreparedData, write_json\n    project_dir,prepared_dir=Path(project_dir),Path(prepared_dir)\n    reference=read(project_dir/\'configs/reference.json\')\n    identity=preparation_identity(project_dir)\n    marker=prepared_dir/\'preparation_complete.json\'\n    if marker.exists():\n        if read(marker)[\'identity\']!=identity:\n            raise RuntimeError(f\'Incompatible preparation cache: {prepared_dir}. Choose a new preparation directory or restore compatible code/dependencies.\')\n        data=PreparedData(prepared_dir,verify=True)\n        reference_check(prepared_dir,reference)\n        print(\'Reusing reference-verified complete preparation\',flush=True)\n        return data.root\n    if prepared_dir.exists():\n        raise RuntimeError(f\'Incomplete preparation exists at {prepared_dir}. Preserve it for diagnosis and use a new path; partial files are never accepted.\')\n    import kagglehub\n    print(\'Acquiring source version 1 directly inside Kaggle...\',flush=True)\n    try:\n        source_dir=Path(kagglehub.dataset_download(SOURCE))\n    except Exception as exc:\n        raise RuntimeError(\'Cannot acquire the public version-1 source. Enable Internet, check Kaggle account/consent access, and rerun. No credentials or alternate dataset are embedded. \'+str(exc)) from exc\n    found=list(source_dir.rglob(\'complaints_processed.csv\'))\n    if len(found)!=1:\n        raise RuntimeError(\'Expected exactly one complaints_processed.csv in source version 1.\')\n    csv_path=verify_csv(found[0])\n    write_json(project_dir/\'reports/source_download.json\',{\n        \'handle\':SOURCE,\'source_csv\':str(csv_path),\'sha256\':CSV_SHA,\n        \'authentication\':\'No explicit login/API key supplied by this notebook; Kaggle session or public access used\',\n        \'kagglehub_version\':importlib.metadata.version(\'kagglehub\')})\n    staging=prepared_dir.with_name(prepared_dir.name+\'.building\')\n    staging.mkdir(parents=True,exist_ok=True)\n    previous=staging/\'build_identity.json\'\n    if previous.exists() and read(previous)!=identity:\n        raise RuntimeError(f\'Incompatible partial preparation at {staging}; use a different preparation path.\')\n    for stale in [\'prepared_manifest.json\', \'reference_comparison.json\', \'preparation_complete.json\']:\n        (staging / stale).unlink(missing_ok=True)\n    write_json(previous,identity)\n    # Rebuild a matching interrupted preparation from scratch; no partial reuse.\n    # Its directory is retained if preparation fails, with diagnostic reports.\n    subprocess.check_call([sys.executable,\'-u\',\'-m\',\'complaints.preparation\',\n                           \'--csv\',str(csv_path),\'--output\',str(staging)],\n                          env={**__import__(\'os\').environ,\'PYTHONPATH\':str(project_dir/\'src\')})\n    data=PreparedData(staging,verify=True)\n    comparison=reference_check(staging,reference)\n    # Independently reconstruct train-only vocabulary and re-tokenize sampled rows.\n    from .verify_preparation import verify\n    verification=verify(staging)\n    del data\n    write_json(staging/\'preparation_complete.json\',{\'identity\':identity,\n        \'reference\':comparison,\'independent_verification\':verification})\n    staging.rename(prepared_dir)\n    print(\'Preparation and independent reference checks passed.\',flush=True)\n    return prepared_dir\n'
target = PROJECT_DIR / 'src/complaints/bootstrap.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/bootstrap.py


In [5]:
# Embedded file: src/complaints/data.py
embedded_source = '"""Strict loading of fully prepared, fingerprinted inputs."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom .text import clean_recurrent, normalize_text, text_hash\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef safe_extract(archive, destination):\n    destination = Path(destination).resolve()\n    with zipfile.ZipFile(archive) as z:\n        for member in z.infolist():\n            target = (destination / member.filename).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError("Archive member escapes destination")\n        z.extractall(destination)\n\n\ndef find_prepared(root="/kaggle/input", extraction_root="/kaggle/working/prepared_input"):\n    root = Path(root)\n    manifests = sorted(root.rglob("prepared_manifest.json")) if root.exists() else []\n    if len(manifests) == 1:\n        return manifests[0].parent\n    if len(manifests) > 1:\n        raise RuntimeError("Multiple prepared datasets found. Set DATA_DIR explicitly in the configuration cell.")\n    candidates = sorted(root.rglob("consumer_complaints_prepared_data.zip")) if root.exists() else []\n    if len(candidates) != 1:\n        raise FileNotFoundError("Attach the supplied consumer_complaints_prepared_data.zip as a Kaggle dataset/input, then rerun. Set DATA_DIR for a custom location.")\n    safe_extract(candidates[0], extraction_root)\n    manifests = list(Path(extraction_root).rglob("prepared_manifest.json"))\n    if len(manifests) != 1:\n        raise RuntimeError("Prepared archive must contain exactly one prepared_manifest.json")\n    return manifests[0].parent\n\n\nclass EncodedDataset(Dataset):\n    def __init__(self, ids, lengths, labels, indices):\n        self.ids, self.lengths, self.labels = ids, lengths, labels\n        self.indices = np.asarray(indices, dtype=np.int64)\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, index):\n        pos = int(self.indices[index])\n        length = int(self.lengths[pos])\n        return self.ids[pos, :length].astype(np.int64), length, int(self.labels[pos]), pos\n\n\ndef collate_encoded(rows):\n    lengths = torch.tensor([row[1] for row in rows], dtype=torch.long)\n    ids = torch.zeros((len(rows), int(lengths.max())), dtype=torch.long)\n    for i, row in enumerate(rows):\n        ids[i, :row[1]] = torch.from_numpy(row[0])\n    return {"input_ids": ids, "lengths": lengths,\n            "labels": torch.tensor([row[2] for row in rows], dtype=torch.long),\n            "indices": torch.tensor([row[3] for row in rows], dtype=torch.long)}\n\n\nclass PreparedData:\n    def __init__(self, root, verify=True):\n        self.root = Path(root)\n        self.manifest = read_json(self.root / "prepared_manifest.json")\n        if self.manifest["schema_version"] != 1:\n            raise ValueError("Unsupported prepared data version")\n        if verify:\n            for relative, expected in self.manifest["files"].items():\n                path = self.root / relative\n                if path.stat().st_size != expected["size_bytes"] or sha256(path) != expected["sha256"]:\n                    raise RuntimeError(f"Prepared input integrity failed: {relative}")\n        self.records = pd.read_csv(self.root / "records.csv.gz", keep_default_na=False)\n        self.encoded = dict(np.load(self.root / "encoded_inputs.npz", allow_pickle=False))\n        self.config = read_json(self.root / "experiment.json")\n        self.preprocessing = read_json(self.root / "preprocessing.json")\n        self.labels = read_json(self.root / "labels.json")\n        self.label_names = [self.labels["id2label"][str(i)] for i in range(len(self.labels["id2label"]))]\n        self.vocabulary = read_json(self.root / "vocabulary.json")\n        self.targets = self.records.label_id.to_numpy(dtype=np.int64)\n        self._validate()\n\n    def _validate(self):\n        records = self.records\n        n = self.manifest["record_count"]\n        assert len(records) == n\n        assert np.array_equal(records.prepared_index.to_numpy(), np.arange(n))\n        assert records.source_id.is_unique\n        records["text_sha256"] = records.narrative.map(lambda text: text_hash(normalize_text(text)))\n        stopwords = set(self.preprocessing["stopwords"])\n        records["recurrent_sha256"] = records.narrative.map(lambda text: text_hash(clean_recurrent(text, stopwords)))\n        for kind in ["recurrent", "transformer"]:\n            records[kind + "_input_sha256"] = [hashlib.sha256(row.astype("<i4").tobytes()).hexdigest() for row in self.encoded[kind + "_ids"]]\n        assert records.text_sha256.is_unique and records.recurrent_sha256.is_unique\n        for column in ["split_group_id", "recurrent_input_sha256", "transformer_input_sha256"]:\n            assert records.groupby(column).split.nunique().max() == 1\n        assert set(records.split) == {"train", "validation", "test"}\n        assert records.narrative.str.strip().ne("").all()\n        assert set(self.targets) == set(range(len(self.label_names)))\n        assert records["product"].map(self.labels["label2id"]).eq(records.label_id).all()\n        for split in ["train", "validation", "test"]:\n            assert set(self.targets[self.indices(split)]) == set(range(len(self.label_names)))\n        for kind, cap in [("recurrent", self.preprocessing["recurrent_max_length"]),\n                          ("transformer", self.preprocessing["transformer_max_length"])]:\n            ids = self.encoded[f"{kind}_ids"]\n            lengths = self.encoded[f"{kind}_lengths"]\n            assert ids.shape == (n, cap)\n            assert lengths.shape == (n,)\n            assert np.all((lengths > 0) & (lengths <= cap))\n            assert np.all(ids >= 0)\n            for start in range(0, n, 4096):\n                block = ids[start:start + 4096]\n                mask = np.arange(cap)[None, :] >= lengths[start:start + 4096, None]\n                assert np.all(block[mask] == 0)\n        assert int(self.encoded["recurrent_ids"].max()) < len(self.vocabulary)\n        counts = np.bincount(self.targets[self.indices("train")], minlength=len(self.label_names))\n        expected_weights = counts.sum() / (len(counts) * counts)\n        assert np.allclose(expected_weights, self.preprocessing["class_weights"])\n\n    def indices(self, split):\n        if split not in {"train", "validation", "test"}:\n            raise ValueError(split)\n        return np.flatnonzero(self.records.split.to_numpy() == split)\n\n    def dataset(self, split, kind, indices=None):\n        prefix = "transformer" if kind == "distilbert" else "recurrent"\n        positions = self.indices(split) if indices is None else indices\n        if not np.isin(positions, self.indices(split)).all():\n            raise ValueError("Dataset indices cross split boundaries")\n        return EncodedDataset(self.encoded[f"{prefix}_ids"], self.encoded[f"{prefix}_lengths"], self.targets, positions)\n'
target = PROJECT_DIR / 'src/complaints/data.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/data.py


In [6]:
# Embedded file: src/complaints/evaluation.py
embedded_source = '"""Metrics, plots and error analysis; test access is orchestrated after selection."""\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support\n\nfrom .data import write_json\n\n\ndef metrics_from_predictions(truth, prediction, names):\n    labels = np.arange(len(names))\n    result = {"accuracy": float(accuracy_score(truth, prediction)), "records": len(truth)}\n    for average in ["macro", "weighted"]:\n        precision, recall, f1, _ = precision_recall_fscore_support(truth, prediction, labels=labels, average=average, zero_division=0)\n        result.update({f"{average}_precision": float(precision), f"{average}_recall": float(recall), f"{average}_f1": float(f1)})\n    result["per_class"] = classification_report(truth, prediction, labels=labels, target_names=names, output_dict=True, zero_division=0)\n    return result\n\n\ndef save_evaluation(output_dir, probabilities, indices, data, loss, name, split):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    truth = data.targets[indices]\n    prediction = probabilities.argmax(axis=1)\n    metrics = metrics_from_predictions(truth, prediction, data.label_names)\n    metrics["loss"] = float(loss)\n    metrics["split"] = split\n    metrics["model"] = name\n    write_json(output_dir / "metrics.json", metrics)\n    pd.DataFrame(metrics["per_class"]).T.to_csv(output_dir / "classification_report.csv")\n    cm = confusion_matrix(truth, prediction, labels=np.arange(len(data.label_names)))\n    normalized = np.divide(cm, cm.sum(axis=1, keepdims=True), out=np.zeros_like(cm, dtype=float), where=cm.sum(axis=1, keepdims=True) != 0)\n    for label, matrix in [("counts", cm), ("normalized", normalized)]:\n        pd.DataFrame(matrix, index=data.label_names, columns=data.label_names).to_csv(output_dir / f"confusion_matrix_{label}.csv")\n    fig, axes = plt.subplots(1, 2, figsize=(13, 5.7))\n    for ax, matrix, title, formatter in [(axes[0], cm, "Counts", "d"), (axes[1], normalized, "Row-normalized", ".2f")]:\n        im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if formatter == ".2f" else None)\n        ax.set_xticks(range(len(data.label_names)), data.label_names, rotation=40, ha="right", fontsize=8)\n        ax.set_yticks(range(len(data.label_names)), data.label_names, fontsize=8)\n        ax.set_xlabel("Predicted")\n        ax.set_ylabel("True")\n        ax.set_title(f"{name} — {split} — {title}")\n        for row in range(len(data.label_names)):\n            for col in range(len(data.label_names)):\n                ax.text(col, row, format(matrix[row, col], formatter), ha="center", va="center", fontsize=8,\n                        color="white" if matrix[row, col] > matrix.max() / 2 else "black")\n        fig.colorbar(im, ax=ax, fraction=.046, pad=.04)\n    fig.tight_layout()\n    fig.savefig(output_dir / "confusion_matrices.png", dpi=150)\n    plt.close(fig)\n    records = data.records.iloc[indices].copy()\n    records["predicted_id"] = prediction\n    records["predicted_label"] = [data.label_names[i] for i in prediction]\n    records["confidence"] = probabilities.max(axis=1)\n    records["correct"] = truth == prediction\n    records["recurrent_truncated"] = data.encoded["recurrent_full_lengths"][indices] > data.preprocessing["recurrent_max_length"]\n    records["transformer_truncated"] = data.encoded["transformer_full_lengths"][indices] > data.preprocessing["transformer_max_length"]\n    records.drop(columns=["narrative"]).to_csv(output_dir / "predictions.csv.gz", index=False)\n    np.savez_compressed(output_dir / "probabilities.npz", probabilities=probabilities, indices=indices)\n    # This is diagnostic output, not another source for selecting the deployment winner.\n    examples = pd.concat([records.loc[~records.correct].sort_values("confidence", ascending=False).head(15),\n                          records.loc[records.correct].sort_values("confidence").head(5)])\n    examples.to_csv(output_dir / "error_and_correct_examples.csv", index=False)\n    strata = {}\n    for col in ["recurrent_truncated", "transformer_truncated"]:\n        strata[col] = {}\n        for value in [False, True]:\n            group = records.loc[records[col].eq(value)]\n            strata[col][str(value)] = {"records": len(group), "accuracy": float(group.correct.mean()) if len(group) else None}\n    write_json(output_dir / "truncation_analysis.json", strata)\n    return metrics\n\n\ndef save_learning_curves(history, output):\n    if not history:\n        return\n    frame = pd.DataFrame(history)\n    frame.to_csv(Path(output) / "history.csv", index=False)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))\n    axes[0].plot(frame.epoch, frame.training_loss, label="Training")\n    axes[0].plot(frame.epoch, frame.validation_loss, label="Validation")\n    axes[0].set_ylabel("Class-weighted loss")\n    axes[0].legend()\n    axes[1].plot(frame.epoch, frame.validation_macro_f1, label="Macro F1")\n    axes[1].plot(frame.epoch, frame.validation_accuracy, label="Accuracy")\n    axes[1].set_ylabel("Validation score")\n    axes[1].legend()\n    for ax in axes:\n        ax.set_xlabel("Epoch")\n        ax.set_xticks(frame.epoch)\n        ax.grid(alpha=.2)\n    fig.tight_layout()\n    fig.savefig(Path(output) / "learning_curves.png", dpi=150)\n    plt.close(fig)\n'
target = PROJECT_DIR / 'src/complaints/evaluation.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/evaluation.py


In [7]:
# Embedded file: src/complaints/inference.py
embedded_source = '"""CPU-capable inference from an exported bundle, without training data."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport torch\n\nfrom .models import RecurrentClassifier\nfrom .text import clean_recurrent, encode_recurrent, normalize_text\n\n\ndef load_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\nclass ComplaintPredictor:\n    def __init__(self, bundle_dir, device="cpu", allow_verification=False):\n        self.root = Path(bundle_dir)\n        self.device = torch.device(device)\n        self.bundle = load_json(self.root / "bundle.json")\n        if self.bundle["training_status"] != "trained":\n            raise ValueError("Bundle is not a trained export")\n        if self.bundle.get("verification_only") and not allow_verification:\n            raise ValueError("Functional-check weights cannot be used as a final trained application.")\n        for relative, expected in self.bundle["file_sha256"].items():\n            target = self.root / relative\n            if not target.exists():\n                if "safetensors" in relative or "model.pt" in relative:\n                    raise FileNotFoundError(\n                        f"Trained model weights missing at \'{target}\'. "\n                        f"On a fresh clone, run \'python scripts/download_model.py\' to fetch and verify them."\n                    )\n                raise FileNotFoundError(f"Required inference file missing: {relative}")\n            with target.open("rb") as f:\n                actual = hashlib.file_digest(f, "sha256").hexdigest()\n            if actual != expected:\n                raise RuntimeError(f"Inference file failed integrity check: {relative}")\n        self.kind = self.bundle["model"]\n        self.preprocessing = load_json(self.root / "preprocessing.json")\n        self.labels = load_json(self.root / "labels.json")\n        self.names = [self.labels["id2label"][str(i)] for i in range(len(self.labels["id2label"]))]\n        if self.kind == "distilbert":\n            from transformers import AutoModelForSequenceClassification, AutoTokenizer\n            self.model = AutoModelForSequenceClassification.from_pretrained(self.root / "model", local_files_only=True, attn_implementation="eager")\n            self.tokenizer = AutoTokenizer.from_pretrained(self.root / "model", local_files_only=True)\n        else:\n            architecture = load_json(self.root / "architecture.json")\n            self.model = RecurrentClassifier(**architecture)\n            self.model.load_state_dict(torch.load(self.root / "model.pt", map_location="cpu", weights_only=True))\n            self.vocabulary = load_json(self.root / "vocabulary.json")\n            self.stopwords = set(self.preprocessing["stopwords"])\n        self.model.to(self.device).eval()\n\n    @torch.inference_mode()\n    def predict_probabilities(self, text):\n        if not isinstance(text, str) or not text.strip():\n            raise ValueError("Enter a complaint before classifying.")\n        if len(text) > 20000:\n            raise ValueError("Please shorten the complaint to 20,000 characters or fewer.")\n        if not any(character.isalpha() for character in text):\n            raise ValueError("Enter a complaint containing words.")\n        if self.kind == "distilbert":\n            batch = self.tokenizer(normalize_text(text), truncation=True,\n                                   max_length=self.preprocessing["transformer_max_length"], return_tensors="pt")\n            batch = {key: value.to(self.device) for key, value in batch.items() if key in {"input_ids", "attention_mask"}}\n            logits = self.model(**batch).logits\n        else:\n            ids = encode_recurrent(text, self.vocabulary, self.stopwords, self.preprocessing["recurrent_max_length"])\n            logits = self.model(torch.tensor([ids], dtype=torch.long, device=self.device), torch.tensor([len(ids)], dtype=torch.long))\n        return logits.float().softmax(dim=-1)[0].cpu().numpy()\n\n    def input_length(self, text):\n        """Untruncated input length and cap, in this model\'s token units."""\n        if self.kind == "distilbert":\n            return len(self.tokenizer(normalize_text(text), truncation=False, verbose=False)["input_ids"]), self.preprocessing["transformer_max_length"]\n        return len(clean_recurrent(text, self.stopwords).split()), self.preprocessing["recurrent_max_length"]\n\n    def predict(self, text):\n        probabilities = self.predict_probabilities(text)\n        return {name: float(probabilities[i]) for i, name in enumerate(self.names)}\n'
target = PROJECT_DIR / 'src/complaints/inference.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/inference.py


In [8]:
# Embedded file: src/complaints/models.py
embedded_source = '"""Comparable single-layer recurrent classifiers and genuine DistilBERT tuning."""\nfrom torch import nn\nfrom torch.nn.utils.rnn import pack_padded_sequence\n\n\nclass RecurrentClassifier(nn.Module):\n    def __init__(self, kind, vocab_size, num_labels, embedding_dim=128, hidden_dim=128, dropout=0.3):\n        super().__init__()\n        if kind not in {"rnn", "lstm", "gru"}:\n            raise ValueError(kind)\n        self.architecture = dict(kind=kind, vocab_size=vocab_size, num_labels=num_labels,\n                                 embedding_dim=embedding_dim, hidden_dim=hidden_dim, dropout=dropout)\n        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)\n        cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]\n        self.recurrent = cls(embedding_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=False)\n        self.dropout = nn.Dropout(dropout)\n        self.classifier = nn.Linear(hidden_dim, num_labels)\n\n    def forward(self, input_ids, lengths):\n        embedded = self.embedding(input_ids)\n        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        _, hidden = self.recurrent(packed)\n        if isinstance(hidden, tuple):\n            hidden = hidden[0]\n        return self.classifier(self.dropout(hidden[-1]))\n\n\ndef create_model(kind, data, config, pretrained=True):\n    if kind != "distilbert":\n        return RecurrentClassifier(kind=kind, vocab_size=len(data.vocabulary), num_labels=len(data.label_names),\n                                   embedding_dim=config["embedding_dim"], hidden_dim=config["hidden_dim"], dropout=config["dropout"])\n    from transformers import AutoConfig, AutoModelForSequenceClassification\n    arguments = dict(num_labels=len(data.label_names), id2label={i: name for i, name in enumerate(data.label_names)},\n                     label2id=data.labels["label2id"])\n    if pretrained:\n        model = AutoModelForSequenceClassification.from_pretrained(\n            config["transformer_checkpoint"], revision=config["transformer_revision"],\n            use_safetensors=True, attn_implementation="eager", **arguments)\n    else:\n        hf_config = AutoConfig.from_pretrained(data.root / "tokenizer", local_files_only=True, **arguments)\n        model = AutoModelForSequenceClassification.from_config(hf_config, attn_implementation="eager")\n    assert all(parameter.requires_grad for parameter in model.parameters()), "Encoder must be fine-tuned"\n    return model\n\n\ndef forward_batch(model, kind, batch, device):\n    ids = batch["input_ids"].to(device, non_blocking=True)\n    if kind == "distilbert":\n        mask = (ids != 0).long()\n        return model(input_ids=ids, attention_mask=mask).logits\n    return model(ids, batch["lengths"])\n'
target = PROJECT_DIR / 'src/complaints/models.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/models.py


In [9]:
# Embedded file: src/complaints/parallel.py
embedded_source = '"""Independent-model GPU scheduling using fresh, isolated Python processes.\n\nThis module deliberately imports neither torch nor transformers. GPU visibility\nis set before the worker interpreter starts, avoiding CUDA-after-fork problems.\n"""\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef _stop(process):\n    if process.poll() is None:\n        process.terminate()\n        try:\n            process.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait(timeout=10)\n\n\ndef schedule_jobs(jobs, gpu_ids, log_dir, poll_seconds=1, heartbeat_seconds=30):\n    """Run at most one job per GPU, returning assignments after every success.\n\n    Jobs are dictionaries containing unique \'name\', list \'command\', and optional\n    environment overrides. A failure stops the remaining workers and raises;\n    it never permits downstream selection/export to run on partial results.\n    """\n    if not gpu_ids or len(set(gpu_ids)) != len(gpu_ids):\n        raise ValueError(\'Provide at least one unique GPU identifier\')\n    if len({job[\'name\'] for job in jobs}) != len(jobs):\n        raise ValueError(\'Job names must be unique\')\n    log_dir = Path(log_dir)\n    log_dir.mkdir(parents=True, exist_ok=True)\n    pending, active, assignments = list(jobs), {}, []\n    handles = []\n    last_heartbeat = time.monotonic()\n    start = time.monotonic()\n    try:\n        while pending or active:\n            for gpu in gpu_ids:\n                if gpu in active or not pending:\n                    continue\n                job = pending.pop(0)\n                name = job[\'name\']\n                if not name or Path(name).name != name or name in {\'.\', \'..\'}:\n                    raise ValueError(\'Job name must be a simple filename\')\n                log_path = log_dir / (name + \'.log\')\n                handle = log_path.open(\'w\', encoding=\'utf-8\')\n                handles.append(handle)\n                environment = dict(os.environ)\n                environment.update(job.get(\'env\', {}))\n                environment.update(CUDA_VISIBLE_DEVICES=str(gpu),\n                                   COMPLAINTS_ASSIGNED_GPU=str(gpu),\n                                   PYTHONUNBUFFERED=\'1\')\n                options = {\'creationflags\': subprocess.CREATE_NO_WINDOW} if os.name == \'nt\' else {}\n                process = subprocess.Popen(job[\'command\'], env=environment,\n                                           stdout=handle, stderr=subprocess.STDOUT, **options)\n                active[gpu] = {\'process\': process, \'job\': job, \'log\': log_path,\n                               \'offset\': 0, \'started\': time.monotonic()}\n                assignments.append({\'model\': name, \'assigned_gpu\': str(gpu),\n                                    \'started_seconds\': time.monotonic() - start,\n                                    \'log\': str(log_path)})\n                print(f\'Started {name} on GPU {gpu}; log: {log_path}\', flush=True)\n            finished = []\n            for gpu, item in active.items():\n                # Files rather than pipes avoid deadlock from full stdout buffers.\n                with item[\'log\'].open(encoding=\'utf-8\', errors=\'replace\') as stream:\n                    stream.seek(item[\'offset\'])\n                    chunk = stream.read(65536)\n                    item[\'offset\'] = stream.tell()\n                if chunk:\n                    print(f"[{item[\'job\'][\'name\']} / GPU {gpu}]\\n{chunk}", end=\'\', flush=True)\n                status = item[\'process\'].poll()\n                if status is None:\n                    continue\n                # Drain any last messages written since the previous read.\n                with item[\'log\'].open(encoding=\'utf-8\', errors=\'replace\') as stream:\n                    stream.seek(item[\'offset\'])\n                    tail = stream.read()\n                if tail:\n                    print(tail, end=\'\', flush=True)\n                if status != 0:\n                    raise RuntimeError(f"{item[\'job\'][\'name\']} failed on GPU {gpu} (exit {status}). "\n                                       f"Inspect {item[\'log\']}; completed models are retained.")\n                elapsed = time.monotonic() - item[\'started\']\n                print(f"Finished {item[\'job\'][\'name\']} on GPU {gpu} in {elapsed / 60:.1f} min", flush=True)\n                for assignment in assignments:\n                    if assignment[\'model\'] == item[\'job\'][\'name\']:\n                        assignment[\'elapsed_seconds\'] = elapsed\n                        assignment[\'status\'] = \'complete\'\n                finished.append(gpu)\n            for gpu in finished:\n                del active[gpu]\n            if active and time.monotonic() - last_heartbeat >= heartbeat_seconds:\n                print(\'Running: \' + \', \'.join(f"{v[\'job\'][\'name\']} on GPU {g}" for g, v in active.items())\n                      + f\'; waiting jobs: {len(pending)}\', flush=True)\n                last_heartbeat = time.monotonic()\n            if pending or active:\n                time.sleep(poll_seconds)\n    finally:\n        # Includes keyboard interrupts and notebook cell cancellation.\n        for item in active.values():\n            _stop(item[\'process\'])\n        for handle in handles:\n            handle.close()\n    return assignments\n\n\ndef train_models_parallel(data, config, output_root, models, gpu_ids, project_dir):\n    from .data import read_json, write_json\n    output_root, project_dir = Path(output_root).resolve(), Path(project_dir).resolve()\n    if not models or len(set(models)) != len(models):\n        raise ValueError(\'Select at least one model, without duplicate names\')\n    allowed = [\'distilbert\', \'rnn\', \'lstm\', \'gru\']\n    if not set(models).issubset(allowed):\n        raise ValueError(\'Unknown model\')\n    config_path = output_root / \'parallel_config.json\'\n    write_json(config_path, config)\n    # Start the generally longer transformer first, alongside the first RNN.\n    order = [kind for kind in allowed if kind in models]\n    inherited = os.environ.get(\'PYTHONPATH\', \'\')\n    jobs = [{\'name\': kind, \'command\': [sys.executable, \'-u\', \'-m\', \'complaints.worker\',\n             \'--model\', kind, \'--data\', str(data.root.resolve()), \'--config\', str(config_path),\n             \'--output\', str(output_root)],\n             \'env\': {\'PYTHONPATH\': str(project_dir / \'src\') + (os.pathsep + inherited if inherited else \'\'),\n                     \'OMP_NUM_THREADS\': \'2\', \'MKL_NUM_THREADS\': \'2\', \'TOKENIZERS_PARALLELISM\': \'false\',\n                     \'USE_TF\': \'0\', \'USE_FLAX\': \'0\'}} for kind in order]\n    assignments = schedule_jobs(jobs, gpu_ids, output_root / \'worker_logs\')\n    write_json(output_root / \'parallel_execution.json\', {\n        \'strategy\': \'independent models; one process per assigned GPU\',\n        \'gpu_ids\': [str(gpu) for gpu in gpu_ids], \'assignments\': assignments,\n        \'effective_transformer_batch_size\': config[\'transformer_batch_size\'] * config[\'gradient_accumulation_steps\'],\n        \'note\': \'No model is replicated across GPUs; batch sizes and learning rates retain their single-model meaning.\'})\n    return {kind: read_json(output_root / \'models\' / kind / \'result.json\') for kind in models}\n'
target = PROJECT_DIR / 'src/complaints/parallel.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/parallel.py


In [10]:
# Embedded file: src/complaints/pipeline.py
embedded_source = '"""Validation selection, one held-out test phase, export and bounded smoke checks."""\nimport gc\nfrom pathlib import Path\nimport shutil\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom .data import read_json, sha256, write_json\nfrom .evaluation import save_evaluation\nfrom .training import environment_record, experiment_signature, load_checkpoint, make_loader, predict_loader, run_model\n\nREQUIRED_MODELS = ["rnn", "lstm", "gru", "distilbert"]\n\n\ndef freeze_selection(data, config, output_root):\n    output_root = Path(output_root)\n    signature, _ = experiment_signature(data, config)\n    results = {}\n    for kind in REQUIRED_MODELS:\n        result = read_json(output_root / "models" / kind / "result.json")\n        if result["verification"] or result["status"] != "complete" or result["signature"] != signature:\n            raise RuntimeError("Final selection requires four compatible full-data runs")\n        if sha256(output_root / "models" / kind / "best.pt") != result["checkpoint_sha256"]:\n            raise RuntimeError("Checkpoint changed after validation")\n        results[kind] = result\n    best_score = max(result["validation_macro_f1"] for result in results.values())\n    tied = [kind for kind in config["tie_priority"] if results[kind]["validation_macro_f1"] >= best_score - config["tie_tolerance"]]\n    winner = tied[0]\n    selection = {"signature": signature, "selected_model": winner, "selection_metric": "validation_macro_f1",\n                 "best_validation_macro_f1": best_score, "selected_validation_macro_f1": results[winner]["validation_macro_f1"],\n                 "tie_tolerance": config["tie_tolerance"], "tie_priority": config["tie_priority"],\n                 "rationale": "Highest validation macro F1; within 0.001 prefer the smaller architecture in the predeclared order.",\n                 "checkpoint_sha256": {kind: result["checkpoint_sha256"] for kind, result in results.items()},\n                 "test_outcomes_used_for_selection": False, "status": "frozen_before_test"}\n    path = output_root / "selection.json"\n    if path.exists() and read_json(path) != selection:\n        raise RuntimeError("Frozen selection differs. Start a separately named experiment; test outcomes must not change the winner.")\n    write_json(path, selection)\n    rows = [{key: result[key] for key in ["model", "validation_macro_f1", "validation_accuracy", "best_epoch", "epochs_run", "training_seconds_including_initialization", "checkpoint_bytes", "trainable_parameters", "inference_batch1_median_ms"]}\n            for result in results.values()]\n    pd.DataFrame(rows).sort_values("validation_macro_f1", ascending=False).to_csv(output_root / "validation_comparison.csv", index=False)\n    print("Frozen deployment selection:", winner, flush=True)\n    return selection\n\n\ndef final_test(data, config, output_root, device):\n    output_root = Path(output_root)\n    selection = read_json(output_root / "selection.json")\n    signature, _ = experiment_signature(data, config)\n    if selection["signature"] != signature or selection["status"] != "frozen_before_test":\n        raise RuntimeError("Freeze compatible validation selection before any test evaluation")\n    completion = output_root / "final_test/complete.json"\n    if completion.exists():\n        saved = read_json(completion)\n        if saved["signature"] != signature:\n            raise RuntimeError("Existing test results have another experiment signature")\n        print("Reusing completed final test phase; no new model selection.", flush=True)\n        return saved\n    weights = torch.tensor(data.preprocessing["class_weights"], dtype=torch.float32, device=device)\n    results = {}\n    for kind in REQUIRED_MODELS:\n        if sha256(output_root / "models" / kind / "best.pt") != selection["checkpoint_sha256"][kind]:\n            raise RuntimeError("Checkpoint changed since validation freeze")\n        destination = output_root / "final_test" / kind\n        marker = destination / "completed.json"\n        if marker.exists():\n            saved = read_json(marker)\n            if saved["signature"] != signature:\n                raise RuntimeError("Partial final-test results have another signature")\n            results[kind] = read_json(destination / "metrics.json")\n            continue\n        model = load_checkpoint(kind, output_root / "models" / kind, data, config, device)\n        loader = make_loader(data, "test", kind, config, device)\n        probabilities, positions, loss = predict_loader(model, kind, loader, device, weights)\n        results[kind] = save_evaluation(destination, probabilities, positions, data, loss, kind, "test")\n        write_json(marker, {"signature": signature})\n        del model\n        gc.collect()\n        if torch.device(device).type == "cuda":\n            torch.cuda.empty_cache()\n    rows = [{"model": kind, **{key: metrics[key] for key in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1"]},\n             "selected_on_validation": kind == selection["selected_model"]} for kind, metrics in results.items()]\n    pd.DataFrame(rows).to_csv(output_root / "final_test/test_comparison.csv", index=False)\n    summary = {"signature": signature, "selected_model": selection["selected_model"], "test_results": results,\n               "policy": "Selection was frozen before this final test phase. Test outcomes do not change the deployment model."}\n    write_json(completion, summary)\n    return summary\n\n\ndef benchmark_bundle(predictor, texts, repeats):\n    # CPU, batch size 1, preprocessing+tokenization+forward pass; warmup excluded.\n    for text in texts[:3]:\n        predictor.predict_probabilities(text)\n    seconds = []\n    for index in range(repeats):\n        start = time.perf_counter()\n        predictor.predict_probabilities(texts[index % len(texts)])\n        seconds.append(time.perf_counter() - start)\n    return {"device": "cpu", "batch_size": 1, "warmup_calls": min(3, len(texts)), "measured_calls": repeats,\n            "median_ms": float(np.median(seconds) * 1000), "p95_ms": float(np.percentile(seconds, 95) * 1000),\n            "includes_preprocessing_and_tokenization": True, "threads": torch.get_num_threads()}\n\n\ndef export_model(kind, data, config, output_root, selection, verification=False):\n    from .inference import ComplaintPredictor\n    output_root = Path(output_root)\n    export = output_root / "deployment_model"\n    if export.exists():\n        shutil.rmtree(export)\n    export.mkdir(parents=True)\n    model = load_checkpoint(kind, output_root / "models" / kind, data, config, "cpu").eval()\n    for filename in ["labels.json", "preprocessing.json"]:\n        shutil.copy2(data.root / filename, export / filename)\n    if kind == "distilbert":\n        model.save_pretrained(export / "model", safe_serialization=True)\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(data.root / "tokenizer", local_files_only=True)\n        tokenizer.save_pretrained(export / "model")\n    else:\n        torch.save(model.state_dict(), export / "model.pt")\n        write_json(export / "architecture.json", model.architecture)\n        shutil.copy2(data.root / "vocabulary.json", export / "vocabulary.json")\n    write_json(export / "selection.json", selection)\n    bundle = {"schema_version": 1, "model": kind, "training_status": "trained", "verification_only": verification,\n              "preparation_id": data.manifest["preparation_id"], "max_input_characters": 20000,\n              "probabilities_are_calibrated": False, "source_text_was_preprocessed_upstream": True,\n              "file_sha256": {str(path.relative_to(export)): sha256(path) for path in sorted(export.rglob("*")) if path.is_file()}}\n    write_json(export / "bundle.json", bundle)\n    predictor = ComplaintPredictor(export, device="cpu", allow_verification=verification)\n    indices = data.indices("validation")[:min(8, len(data.indices("validation")))]\n    loader = make_loader(data, "validation", kind, config, torch.device("cpu"), indices)\n    weights = torch.tensor(data.preprocessing["class_weights"], dtype=torch.float32)\n    expected, positions, _ = predict_loader(model, kind, loader, torch.device("cpu"), weights)\n    actual = np.stack([predictor.predict_probabilities(data.records.iloc[index].narrative) for index in positions])\n    np.testing.assert_allclose(actual, expected, rtol=2e-4, atol=2e-5)\n    measured = benchmark_bundle(predictor, data.records.iloc[indices].narrative.tolist(), config["benchmark_repeats"])\n    measured["recurrent_max_length"] = config["recurrent_max_length"]\n    measured["transformer_max_length"] = config["transformer_max_length"]\n    measured["validation_reload_max_probability_difference"] = float(np.abs(actual - expected).max())\n    write_json(output_root / "cpu_inference_benchmark.json", measured)\n    del model, predictor\n    gc.collect()\n    return export\n\n\ndef export_selected(data, config, output_root):\n    output_root = Path(output_root)\n    selection = read_json(output_root / "selection.json")\n    signature, _ = experiment_signature(data, config)\n    if selection["signature"] != signature:\n        raise RuntimeError("Selection signature mismatch")\n    if config["final_test"] and not (output_root / "final_test/complete.json").exists():\n        raise RuntimeError("Complete the final test phase before final delivery")\n    export = export_model(selection["selected_model"], data, config, output_root, selection)\n    # Standalone app package, without source data or optimizer/cache files.\n    app_root = output_root / "gradio_app"\n    if app_root.exists():\n        shutil.rmtree(app_root)\n    app_root.mkdir()\n    shutil.copytree(export, app_root / "deployment_model")\n    source_root = Path(__file__).resolve().parents[2]\n    shutil.copy2(source_root / "app.py", app_root / "app.py")\n    shutil.copy2(source_root / "requirements.txt", app_root / "requirements.txt")\n    package_dir = app_root / "src/complaints"\n    package_dir.mkdir(parents=True)\n    for name in ["__init__.py", "text.py", "models.py", "inference.py"]:\n        shutil.copy2(Path(__file__).parent / name, package_dir / name)\n    (app_root / "README.md").write_text(\n        "---\\ntitle: Consumer Complaint Classifier\\nsdk: gradio\\nsdk_version: 5.49.1\\napp_file: app.py\\n---\\n\\n"\n        "# Consumer Complaint Classifier\\n\\nRun `pip install -r requirements.txt` then `python app.py`. "\n        "The selected model and exact inference preprocessing are included. No training at startup.\\n",\n        encoding="utf-8")\n    shutil.make_archive(str(output_root / "consumer_complaints_gradio_app"), "zip", app_root)\n    # Retain all four best checkpoints, reports, code and run configuration for reproducibility.\n    for destination, subdir in [("training_source", "src")]:\n        target = output_root / destination\n        if target.exists():\n            shutil.rmtree(target)\n        shutil.copytree(source_root / subdir, target, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))\n    write_json(output_root / "experiment_config.json", config)\n    write_json(output_root / "environment.json", environment_record())\n    write_json(output_root / "prepared_input_reference.json", data.manifest)\n    # Archive paths are explicit: never accidentally include the archive itself or the HF cache.\n    archive = output_root.parent / f"{output_root.name}_results.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:\n        for path in sorted(output_root.rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts and path.name != "consumer_complaints_gradio_app.zip":\n                if any(part in path.relative_to(output_root).parts for part in ["gradio_app", "deployment_model", "verification_only"]):\n                    continue\n                z.write(path, str(path.relative_to(output_root)))\n        z.write(output_root / "consumer_complaints_gradio_app.zip", "consumer_complaints_gradio_app.zip")\n    reports_archive = output_root.parent / f"{output_root.name}_reports.zip"\n    with zipfile.ZipFile(reports_archive, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(output_root.rglob("*")):\n            relative = path.relative_to(output_root)\n            if path.is_file() and path.suffix not in {".pt", ".zip", ".pyc"} and not any(\n                part in relative.parts for part in ["gradio_app", "deployment_model", "verification_only", "__pycache__"]\n            ):\n                z.write(path, str(relative))\n    return {"reports_archive": str(reports_archive), "results_archive": str(archive),\n            "app_archive": str(output_root / "consumer_complaints_gradio_app.zip")}\n\n\ndef verification_indices(data, split, per_class=2):\n    indices = []\n    allowed = data.indices(split)\n    for label in range(len(data.label_names)):\n        indices.extend(allowed[data.targets[allowed] == label][:per_class].tolist())\n    return np.array(sorted(indices), dtype=np.int64)\n\n\ndef run_verification(data, config, output_root, device, kinds=None):\n    kinds = REQUIRED_MODELS if kinds is None else list(kinds)\n    smoke = {**config, "recurrent_epochs": 1, "transformer_epochs": 1, "recurrent_batch_size": 5,\n             "transformer_batch_size": 2, "eval_batch_size": 5, "gradient_accumulation_steps": 1,\n             "benchmark_repeats": 3, "warmup_ratio": 0.0}\n    root = Path(output_root) / "verification_only"\n    results = {}\n    for kind in kinds:\n        results[kind] = run_model(kind, data, smoke, root / kind, torch.device(device), verification=True,\n                                  train_indices=verification_indices(data, "train", 1),\n                                  validation_indices=verification_indices(data, "validation", 1))\n        export_model(kind, data, smoke, root / kind, {"verification_only": True, "model": kind}, verification=True)\n    write_json(root / "verification_summary.json", {"status": "passed", "verification_only": True, "models": results,\n                                                    "note": "Tiny functional checks, not experimental scores."})\n    write_json(Path(output_root) / "functional_checks.json", {"status": "passed", "verification_only": True,\n                                                            "models_checked": list(results), "note": "Tiny checks only; verification weights discarded."})\n    shutil.rmtree(root)\n    return results\n'
target = PROJECT_DIR / 'src/complaints/pipeline.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/pipeline.py


In [11]:
# Embedded file: src/complaints/preparation.py
embedded_source = '"""Reproduce audited preparation inside Kaggle from a verified CSV."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport sys\nimport urllib.request\nimport zipfile\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom transformers import AutoConfig, AutoTokenizer\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT / "src"))\nfrom complaints.text import clean_recurrent, normalize_text, text_hash, PREPROCESSING_VERSION\n\nSOURCE_SHA256 = "45e406e64bba3a77dc76a66b096eb54c7b5e5a896be4dd851503e505465c6599"\nSOURCE_SLUG = "shashwatwork/consume-complaints-dataset-fo-nlp"\n\n\ndef dump(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef sha(path):\n    with path.open("rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--csv", type=Path, required=True)\n    parser.add_argument("--output", type=Path, default=ROOT / "prepared_data")\n    args = parser.parse_args()\n    out = args.output.resolve()\n    out.mkdir(parents=True, exist_ok=True)\n    cfg = json.loads((ROOT / "configs/experiment.json").read_text())\n    (ROOT / "reports/figures").mkdir(parents=True, exist_ok=True)\n    source_csv_sha = sha(args.csv)\n    if source_csv_sha != "a120ac8450cb3839bf0085bbae68f7c0f3b5351302cafd0bc85ae27a251010b4":\n        raise RuntimeError("Source CSV differs from audited version 1; preparation stopped.")\n    raw = pd.read_csv(args.csv, keep_default_na=False)\n    if raw.columns.tolist() != ["Unnamed: 0", "product", "narrative"]:\n        raise RuntimeError("Unexpected source schema")\n    frame = raw.rename(columns={"Unnamed: 0": "source_id"}).copy()\n    if not frame.source_id.is_unique:\n        raise RuntimeError("Source row IDs are not unique")\n    frame["narrative"] = frame.narrative.map(normalize_text)\n    frame["product"] = frame["product"].astype(str).str.strip()\n    frame["text_sha256"] = frame.narrative.map(text_hash)\n    stopwords = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never", "none", "cannot", "without"}\n    frame["recurrent_text"] = frame.narrative.map(lambda t: clean_recurrent(t, stopwords))\n    frame["recurrent_sha256"] = frame.recurrent_text.map(text_hash)\n    frame["reason"] = "retained"\n    frame["canonical_source_id"] = frame.source_id\n    counts = [{"stage": "source", "remaining": len(frame), "excluded": 0}]\n\n    def exclude(mask, reason):\n        mask = mask & frame.reason.eq("retained")\n        n = int(mask.sum())\n        frame.loc[mask, "reason"] = reason\n        counts.append({"stage": reason, "remaining": int(frame.reason.eq("retained").sum()), "excluded": n})\n\n    exclude(frame.narrative.eq("") | frame["product"].eq(""), "blank_text_or_label")\n    active = frame.loc[frame.reason.eq("retained")]\n    conflicts = active.groupby("text_sha256")["product"].nunique()\n    conflicting_keys = set(conflicts[conflicts.gt(1)].index)\n    exclude(frame.text_sha256.isin(conflicting_keys), "conflicting_source_text_labels")\n    active = frame.loc[frame.reason.eq("retained")]\n    canonical = active.groupby("text_sha256").source_id.min()\n    frame.loc[active.index, "canonical_source_id"] = active.text_sha256.map(canonical)\n    exclude(frame.reason.eq("retained") & frame.source_id.ne(frame.canonical_source_id), "duplicate_source_text")\n    exclude(frame.recurrent_text.eq(""), "empty_after_recurrent_cleaning")\n    active = frame.loc[frame.reason.eq("retained")]\n    rec_conflicts = active.groupby("recurrent_sha256")["product"].nunique()\n    rec_conflict_keys = set(rec_conflicts[rec_conflicts.gt(1)].index)\n    exclude(frame.recurrent_sha256.isin(rec_conflict_keys), "conflicting_recurrent_text_labels")\n    active = frame.loc[frame.reason.eq("retained")]\n    rec_canonical = active.groupby("recurrent_sha256").source_id.min()\n    frame.loc[active.index, "canonical_source_id"] = active.recurrent_sha256.map(rec_canonical)\n    exclude(frame.reason.eq("retained") & frame.source_id.ne(frame.canonical_source_id), "duplicate_recurrent_text")\n    data = frame.loc[frame.reason.eq("retained")].sort_values("source_id").copy().reset_index(drop=True)\n    labels = sorted(data["product"].unique().tolist())\n    label2id = {label: i for i, label in enumerate(labels)}\n    data["label_id"] = data["product"].map(label2id)\n    tokenizer = AutoTokenizer.from_pretrained(cfg["transformer_checkpoint"], revision=cfg["transformer_revision"])\n    tokenizer.save_pretrained(out / "tokenizer")\n    AutoConfig.from_pretrained(cfg["transformer_checkpoint"], revision=cfg["transformer_revision"]).save_pretrained(out / "tokenizer")\n    transformer = np.zeros((len(data), cfg["transformer_max_length"]), dtype=np.uint16)\n    hf_lengths = np.zeros(len(data), dtype=np.uint16)\n    hf_full_lengths = np.zeros(len(data), dtype=np.int32)\n    for start in range(0, len(data), 1024):\n        texts = data.narrative.iloc[start:start + 1024].tolist()\n        all_ids = tokenizer(texts, truncation=False, padding=False, verbose=False)["input_ids"]\n        enc = tokenizer(texts, truncation=True, max_length=cfg["transformer_max_length"], padding="max_length", return_tensors="np")\n        end = start + len(texts)\n        transformer[start:end] = enc["input_ids"].astype(np.uint16)\n        hf_lengths[start:end] = enc["attention_mask"].sum(axis=1).astype(np.uint16)\n        hf_full_lengths[start:end] = [len(ids) for ids in all_ids]\n        if start % 20480 == 0:\n            print(f"Tokenized {end:,}/{len(data):,}", flush=True)\n    # Group complaints that become identical after either model\'s head truncation.\n    # These keys use fixed text rules / a fixed pretrained tokenizer, not learned vocabulary.\n    parents = np.arange(len(data))\n    def find(x):\n        while parents[x] != x:\n            parents[x] = parents[parents[x]]\n            x = parents[x]\n        return int(x)\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b:\n            parents[max(a, b)] = min(a, b)\n    prefix_keys = data.recurrent_text.map(lambda text: text_hash(" ".join(text.split()[:cfg["recurrent_max_length"]])))\n    hf_keys = [hashlib.sha256(row.astype("<i4").tobytes()).hexdigest() for row in transformer]\n    for keys in [prefix_keys, hf_keys]:\n        seen = {}\n        for index, key in enumerate(keys):\n            if key in seen:\n                union(index, seen[key])\n            else:\n                seen[key] = index\n    # Unknown-token substitution can create further equal inputs. Monotonically\n    # merge those groups, re-split and refit the vocabulary from the new training\n    # partition until actual encoded inputs are isolated. No vocabulary fitted\n    # on a provisional partition is ever shipped or used for training.\n    for split_pass in range(10):\n        data["split_group_id"] = [int(data.source_id.iloc[find(i)]) for i in range(len(data))]\n        folds = np.full(len(data), -1, dtype=np.int8)\n        splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=cfg["seed"])\n        for fold, (_, indices) in enumerate(splitter.split(np.zeros(len(data)), data.label_id, groups=data.split_group_id)):\n            folds[indices] = fold\n        data["split"] = np.where(folds == 0, "validation", np.where(folds == 1, "test", "train"))\n        train = data.loc[data.split.eq("train")]\n        counter = Counter(word for text in train.recurrent_text for word in text.split())\n        ordered = sorted(counter.items(), key=lambda item: (-item[1], item[0]))\n        vocabulary = {"<PAD>": 0, "<UNK>": 1}\n        vocabulary.update({word: i + 2 for i, (word, _) in enumerate(ordered[:cfg["vocabulary_cap"] - 2])})\n        recurrent = np.zeros((len(data), cfg["recurrent_max_length"]), dtype=np.uint16)\n        rec_lengths = np.zeros(len(data), dtype=np.uint16)\n        rec_full_lengths = np.zeros(len(data), dtype=np.int32)\n        unk_counts = {split: 0 for split in ["train", "validation", "test"]}\n        word_counts = {split: 0 for split in unk_counts}\n        for i, row in enumerate(data.itertuples()):\n            words = row.recurrent_text.split()\n            ids = [vocabulary.get(word, 1) for word in words]\n            rec_full_lengths[i] = len(ids)\n            ids = ids[:cfg["recurrent_max_length"]]\n            rec_lengths[i] = len(ids)\n            recurrent[i, :len(ids)] = ids\n            unk_counts[row.split] += sum(x == 1 for x in ids)\n            word_counts[row.split] += len(ids)\n        data["recurrent_input_sha256"] = [hashlib.sha256(row.astype("<i4").tobytes()).hexdigest() for row in recurrent]\n        cross_split = data.groupby("recurrent_input_sha256").split.nunique().gt(1)\n        print(f"Split verification pass {split_pass + 1}: {int(cross_split.sum())} recurrent input collisions across splits", flush=True)\n        if not cross_split.any():\n            break\n        seen = {}\n        for index, key in enumerate(data.recurrent_input_sha256):\n            if key in seen:\n                union(index, seen[key])\n            else:\n                seen[key] = index\n    else:\n        raise RuntimeError("Encoded input grouping did not converge; no prepared split is accepted")\n    data["prepared_index"] = np.arange(len(data))\n    split_by_id = data.set_index("source_id")["split"]\n    frame["split"] = frame.source_id.map(split_by_id).fillna("")\n    for column in ["text_sha256", "recurrent_sha256"]:\n        assert data[column].is_unique, column\n    data["transformer_input_sha256"] = hf_keys\n    for column in ["split_group_id", "recurrent_input_sha256", "transformer_input_sha256"]:\n        assert data.groupby(column).split.nunique().max() == 1, f"Input overlap across splits: {column}"\n    np.savez_compressed(out / "encoded_inputs.npz", recurrent_ids=recurrent, recurrent_lengths=rec_lengths,\n                        transformer_ids=transformer, transformer_lengths=hf_lengths,\n                        recurrent_full_lengths=rec_full_lengths, transformer_full_lengths=hf_full_lengths)\n    # Recompute fingerprints from verified bytes at load time rather than ship\n    # 496k redundant SHA strings. This reduces handoff bandwidth substantially.\n    keep = ["prepared_index", "source_id", "product", "label_id", "split", "split_group_id", "narrative"]\n    data[keep].to_csv(out / "records.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})\n    ledger_columns = ["source_id", "product", "reason", "canonical_source_id", "split"]\n    frame[ledger_columns].to_csv(out / "source_row_ledger.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})\n    split_counts = pd.crosstab(data["product"], data.split).reindex(labels)\n    split_counts.to_csv(ROOT / "reports/class_counts_by_split.csv")\n    class_counts = np.bincount(train.label_id, minlength=len(labels))\n    weights = len(train) / (len(labels) * class_counts)\n    dump(out / "labels.json", {"label2id": label2id, "id2label": {str(i): name for name, i in label2id.items()}})\n    dump(out / "vocabulary.json", vocabulary)\n    dump(out / "preprocessing.json", {\n        "version": PREPROCESSING_VERSION, "stopwords": sorted(stopwords), "negations_preserved": ["no", "not", "nor", "never", "none", "cannot", "without"],\n        "source_text": "Already processed upstream; lowercase alphabetic words only. Raw narratives unavailable.",\n        "recurrent_pipeline": "NFKC, casefold, replace non a-z with spaces, fixed stopwords, training-only vocabulary, head truncation",\n        "transformer_pipeline": "NFKC/casefold/whitespace source normalization only, pinned pretrained tokenizer, head truncation",\n        "class_weights": weights.tolist(), "class_weight_policy": "N / (K * training_class_count)",\n        "recurrent_max_length": cfg["recurrent_max_length"], "transformer_max_length": cfg["transformer_max_length"],\n    })\n    dump(out / "experiment.json", cfg)\n    length_summary = {}\n    for split in ["train", "validation", "test"]:\n        mask = data.split.eq(split).to_numpy()\n        length_summary[split] = {\n            "records": int(mask.sum()),\n            "recurrent_length_percentiles": {str(q): float(np.percentile(rec_full_lengths[mask], q)) for q in [50, 90, 95, 99, 100]},\n            "transformer_length_percentiles": {str(q): float(np.percentile(hf_full_lengths[mask], q)) for q in [50, 90, 95, 99, 100]},\n            "recurrent_truncation_fraction": float(np.mean(rec_full_lengths[mask] > cfg["recurrent_max_length"])),\n            "transformer_truncation_fraction": float(np.mean(hf_full_lengths[mask] > cfg["transformer_max_length"])),\n            "recurrent_unknown_token_fraction": unk_counts[split] / max(word_counts[split], 1),\n        }\n    source_counts = frame["product"].value_counts().to_dict()\n    audit = {\n        "source": SOURCE_SLUG, "source_version": 1, "retrieved_date": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).date().isoformat(), "historical_source_archive_sha256": SOURCE_SHA256,\n        "source_csv_sha256": source_csv_sha, "source_rows": len(raw), "source_columns": raw.columns.tolist(),\n        "source_class_counts": source_counts, "official_splits": "None: one CSV with no split column",\n        "normalized_conflicting_groups": len(conflicting_keys), "recurrent_conflicting_groups_after_source_dedup": len(rec_conflict_keys),\n        "exclusion_flow": counts, "retained_rows": len(data), "labels": labels,\n        "split_counts": {k: int(v) for k, v in data.split.value_counts().items()},\n        "class_counts_by_split": split_counts.to_dict(), "lengths": length_summary,\n        "vocabulary_size_including_specials": len(vocabulary), "training_distinct_words": len(counter),\n        "training_class_weights": {label: float(weights[i]) for i, label in enumerate(labels)},\n        "duplicate_overlap_between_splits": {"normalized_source_text": 0, "recurrent_cleaned_text": 0, "recurrent_encoded_input": 0, "transformer_encoded_input": 0},\n        "split_policy": "10 StratifiedGroupKFold folds with seed 42: fold 0 validation, fold 1 test, other folds train. Truncated-input equivalence groups stay together.",\n        "split_group_count": int(data.split_group_id.nunique()),\n        "multirow_split_groups": int(data.groupby("split_group_id").size().gt(1).sum()),\n        "encoded_collision_grouping_passes": split_pass + 1,\n        "source_preprocessing_evidence": {\n            "uppercase_narratives": int(raw.narrative.str.contains(r"[A-Z]", regex=True).sum()),\n            "digit_narratives": int(raw.narrative.str.contains(r"\\d", regex=True).sum()),\n            "non_lowercase_letter_or_whitespace_narratives": int(raw.narrative.str.contains(r"[^a-z\\s]", regex=True).sum()),\n            "raw_narratives_available": False,\n        },\n        "near_duplicate_policy": "Exact normalized and recurrent-cleaned duplicates removed. Semantic/paraphrase near-duplicates are not guaranteed absent.",\n        "upstream_limitations": "Unknown upstream cleaning/label consolidation; no raw text or original complaint identifiers. No original-data linkage is invented.",\n        "training_status": "NOT TRAINED",\n        "preparation_versions": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scikit-learn", "transformers", "tokenizers"]},\n    }\n    dump(out / "audit.json", audit)\n    dump(ROOT / "reports/dataset_audit.json", audit)\n    train.groupby("product", sort=True).head(2)[["source_id", "product", "narrative"]].to_csv(ROOT / "reports/training_examples.csv", index=False)\n    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))\n    split_counts[["train", "validation", "test"]].plot.bar(stacked=True, ax=axes[0], color=["#2563eb", "#14b8a6", "#f59e0b"])\n    axes[0].set_title("Retained complaints by category and split")\n    axes[0].set_xlabel("")\n    axes[0].tick_params(axis="x", rotation=25, labelsize=8)\n    axes[0].set_ylabel("Complaints")\n    train_mask = data.split.eq("train").to_numpy()\n    axes[1].hist(hf_full_lengths[train_mask], bins=np.arange(0, 801, 20), color="#2563eb", alpha=.8)\n    axes[1].axvline(256, color="#dc2626", linestyle="--", label="256-token cap")\n    axes[1].set_title("Training narrative lengths (DistilBERT tokens)")\n    axes[1].set_xlabel("Tokens including special tokens; view capped at 800")\n    axes[1].set_ylabel("Complaints")\n    axes[1].legend()\n    fig.tight_layout()\n    fig.savefig(ROOT / "reports/figures/data_overview.png", dpi=170)\n    plt.close(fig)\n    files = {str(p.relative_to(out)): {"sha256": sha(p), "size_bytes": p.stat().st_size}\n             for p in sorted(out.rglob("*")) if p.is_file() and p.name != "prepared_manifest.json"}\n    identity = hashlib.sha256(json.dumps(files, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n    manifest = {"schema_version": 1, "preparation_id": identity, "historical_source_archive_sha256": SOURCE_SHA256,\n                "record_count": len(data), "preprocessing_version": PREPROCESSING_VERSION, "files": files}\n    dump(out / "prepared_manifest.json", manifest)\n    print(json.dumps({"retained": len(data), "splits": audit["split_counts"], "preparation_id": identity,\n                      "lengths": length_summary, "exclusion_flow": counts}, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
target = PROJECT_DIR / 'src/complaints/preparation.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/preparation.py


In [12]:
# Embedded file: src/complaints/text.py
embedded_source = '"""Versioned text normalization shared by preparation and inference."""\nimport hashlib\nimport re\nimport unicodedata\n\nPREPROCESSING_VERSION = "complaints-v1"\nPAD_ID = 0\nUNK_ID = 1\n\n\ndef normalize_text(text):\n    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())\n\n\ndef clean_recurrent(text, stopwords):\n    text = re.sub(r"[^a-z]+", " ", normalize_text(text))\n    return " ".join(word for word in text.split() if word not in stopwords)\n\n\ndef text_hash(text):\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef encode_recurrent(text, vocabulary, stopwords, max_length):\n    words = clean_recurrent(text, stopwords).split()\n    if not words:\n        raise ValueError("No usable words remain after preprocessing.")\n    return [vocabulary.get(word, UNK_ID) for word in words[:max_length]]\n'
target = PROJECT_DIR / 'src/complaints/text.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/text.py


In [13]:
# Embedded file: src/complaints/training.py
embedded_source = '"""Explicit PyTorch training loops; compatible completed runs are reusable."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport random\nimport shutil\nimport time\nimport sys\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom .data import collate_encoded, read_json, sha256, write_json\nfrom .evaluation import metrics_from_predictions, save_evaluation, save_learning_curves\nfrom .models import create_model, forward_batch\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    # Repeated runs can still differ across hardware/CUDA/library versions.\n\n\ndef environment_record():\n    versions = {}\n    for name in ["torch", "transformers", "tokenizers", "numpy", "pandas", "scikit-learn", "safetensors"]:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    return {"python": platform.python_version(), "platform": platform.platform(), "versions": versions,\n            "cuda_version": torch.version.cuda, "cuda_available": torch.cuda.is_available(),\n            "gpus": [{"index": i, "name": torch.cuda.get_device_name(i),\n                      "total_memory_bytes": torch.cuda.get_device_properties(i).total_memory}\n                     for i in range(torch.cuda.device_count())]}\n\n\ndef experiment_signature(data, config):\n    code = hashlib.sha256()\n    for path in sorted(Path(__file__).parent.glob("*.py")):\n        code.update(path.name.encode())\n        code.update(path.read_bytes())\n    payload = {"preparation_id": data.manifest["preparation_id"], "config": config,\n               "code_sha256": code.hexdigest(), "versions": environment_record()["versions"]}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest(), payload\n\n\ndef make_loader(data, split, kind, config, device, indices=None, shuffle=False):\n    batch_size = (config["transformer_batch_size"] if kind == "distilbert" else config["recurrent_batch_size"]) if shuffle else config["eval_batch_size"]\n    generator = torch.Generator().manual_seed(config["seed"])\n    return DataLoader(data.dataset(split, kind, indices), batch_size=batch_size, shuffle=shuffle,\n                      collate_fn=collate_encoded, num_workers=config["num_workers"],\n                      pin_memory=device.type == "cuda", generator=generator)\n\n\n@torch.inference_mode()\ndef predict_loader(model, kind, loader, device, weights):\n    model.eval()\n    probabilities, positions = [], []\n    total_loss, total_weight = 0.0, 0.0\n    for batch in loader:\n        targets = batch["labels"].to(device)\n        logits = forward_batch(model, kind, batch, device).float()\n        losses = nn.functional.cross_entropy(logits, targets, weight=weights, reduction="sum")\n        total_loss += float(losses.item())\n        total_weight += float(weights[targets].sum().item())\n        probabilities.append(logits.softmax(dim=-1).cpu().numpy())\n        positions.append(batch["indices"].numpy())\n    return np.concatenate(probabilities), np.concatenate(positions), total_loss / total_weight\n\n\ndef load_checkpoint(kind, directory, data, config, device):\n    checkpoint = torch.load(Path(directory) / "best.pt", map_location="cpu", weights_only=True)\n    model = create_model(kind, data, config, pretrained=False)\n    model.load_state_dict(checkpoint["model_state"])\n    return model.to(device)\n\n\n@torch.inference_mode()\ndef benchmark_forward(model, kind, data, config, device):\n    """Common validation examples, batch 1; precomputed IDs, transfer+forward only."""\n    candidates = data.indices("validation")[:8]\n    dataset = data.dataset("validation", kind, candidates)\n    batches = [collate_encoded([dataset[i]]) for i in range(len(dataset))]\n    model.eval()\n    for batch in batches[:3]:\n        forward_batch(model, kind, batch, device)\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    timings = []\n    for repeat in range(config["benchmark_repeats"]):\n        start = time.perf_counter()\n        forward_batch(model, kind, batches[repeat % len(batches)], device)\n        if device.type == "cuda":\n            torch.cuda.synchronize(device)\n        timings.append((time.perf_counter() - start) * 1000)\n    return {"device": str(device), "batch_size": 1, "median_ms": float(np.median(timings)),\n            "p95_ms": float(np.percentile(timings, 95)), "warmup_calls": min(3, len(batches)),\n            "measured_calls": config["benchmark_repeats"], "includes_host_to_device_transfer": True,\n            "includes_tokenization": False, "inputs": "Same first eight validation records; each model\'s prepared tokenization",\n            "recurrent_max_length": config["recurrent_max_length"], "transformer_max_length": config["transformer_max_length"]}\n\n\ndef run_model(kind, data, config, output_root, device, verification=False, train_indices=None, validation_indices=None):\n    if kind not in {"rnn", "lstm", "gru", "distilbert"}:\n        raise ValueError(kind)\n    device = torch.device(device)\n    signature, signature_payload = experiment_signature(data, config)\n    output = Path(output_root) / "models" / kind\n    completed = output / "result.json"\n    if completed.exists():\n        result = read_json(completed)\n        if result["signature"] != signature or result["verification"] != verification:\n            raise RuntimeError(f"{kind}: existing outputs belong to a different experiment. Choose a new OUTPUT_DIR; existing results will not be overwritten.")\n        if sha256(output / "best.pt") != result["checkpoint_sha256"]:\n            raise RuntimeError(f"{kind}: checkpoint checksum failed")\n        print(f"{kind}: reusing compatible completed run", flush=True)\n        return result\n    if output.exists() and any(output.iterdir()):\n        previous = read_json(output / "run_identity.json") if (output / "run_identity.json").exists() else None\n        if previous and previous["signature"] != signature:\n            raise RuntimeError(f"{kind}: partial outputs have a different configuration. Choose a new OUTPUT_DIR.")\n        print(f"{kind}: restarting an incomplete run from epoch 1 (weights-only checkpoints do not resume optimization).", flush=True)\n        shutil.rmtree(output)\n    output.mkdir(parents=True, exist_ok=True)\n    write_json(output / "run_identity.json", {"signature": signature, "verification": verification, "identity": signature_payload})\n    seed_everything(config["seed"])\n    start = time.perf_counter()\n    model = create_model(kind, data, config, pretrained=True).to(device)\n    loader = make_loader(data, "train", kind, config, device, train_indices, shuffle=True)\n    validation = make_loader(data, "validation", kind, config, device, validation_indices)\n    weights = torch.tensor(data.preprocessing["class_weights"], dtype=torch.float32, device=device)\n    if kind == "distilbert":\n        no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")\n        groups = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": config["transformer_weight_decay"]},\n                  {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0}]\n        optimizer = torch.optim.AdamW(groups, lr=config["transformer_lr"])\n        from transformers import get_linear_schedule_with_warmup\n        epochs = config["transformer_epochs"]\n        accumulation = config["gradient_accumulation_steps"]\n        updates = math.ceil(len(loader) / accumulation) * epochs\n        scheduler = get_linear_schedule_with_warmup(optimizer, int(updates * config["warmup_ratio"]), updates)\n    else:\n        optimizer = torch.optim.Adam(model.parameters(), lr=config["recurrent_lr"])\n        epochs = config["recurrent_epochs"]\n        accumulation = 1\n        scheduler = None\n    use_amp = config["mixed_precision"] and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    best_f1, best_epoch, stale, monitor_best = -1.0, 0, 0, -1.0\n    history = []\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n    first_batches = []\n    for epoch in range(1, epochs + 1):\n        epoch_start = time.perf_counter()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        total_loss, total_weight = 0.0, 0.0\n        for step, batch in enumerate(tqdm(loader, desc=f"{kind} epoch {epoch}/{epochs}", leave=False, disable=not sys.stderr.isatty())):\n            batch_start = time.perf_counter()\n            targets = batch["labels"].to(device, non_blocking=True)\n            group_start = (step // accumulation) * accumulation\n            group_size = min(accumulation, len(loader) - group_start)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits = forward_batch(model, kind, batch, device)\n                loss_sum = nn.functional.cross_entropy(logits.float(), targets, weight=weights, reduction="sum")\n                weight_sum = weights[targets].sum()\n                loss = loss_sum / weight_sum\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"{kind}: nonfinite training loss; no successful run is recorded")\n            scaler.scale(loss / group_size).backward()\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])\n                previous_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scheduler is not None and scaler.get_scale() >= previous_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            total_loss += float(loss_sum.detach().item())\n            total_weight += float(weight_sum.item())\n            if epoch == 1 and step < 20:\n                if device.type == "cuda":\n                    torch.cuda.synchronize(device)\n                first_batches.append(time.perf_counter() - batch_start)\n                if step == 4:\n                    seconds = float(np.mean(first_batches[1:]))\n                    estimate = seconds * len(loader) * epochs / 60\n                    print(f"\\n{kind} pilot: {seconds:.3f} s/batch; about {estimate:.1f} min for {epochs} training epochs, excluding validation/downloads.", flush=True)\n                    write_json(output / "pilot.json", {"seconds_per_batch_initial": seconds, "batches_per_epoch": len(loader),\n                                                        "maximum_training_minutes_excluding_validation": estimate, "device": str(device)})\n        probabilities, positions, validation_loss = predict_loader(model, kind, validation, device, weights)\n        metrics = metrics_from_predictions(data.targets[positions], probabilities.argmax(axis=1), data.label_names)\n        score = metrics["macro_f1"]\n        history.append({"epoch": epoch, "training_loss": total_loss / total_weight, "validation_loss": validation_loss,\n                        "validation_macro_f1": score, "validation_accuracy": metrics["accuracy"],\n                        "epoch_seconds": time.perf_counter() - epoch_start, "learning_rate": optimizer.param_groups[0]["lr"]})\n        write_json(output / "history.json", history)\n        save_learning_curves(history, output)\n        if score > best_f1:\n            best_f1, best_epoch = score, epoch\n            checkpoint = {"model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                          "kind": kind, "epoch": epoch, "validation_macro_f1": score, "signature": signature,\n                          "verification": verification}\n            temporary = output / "best.pt.tmp"\n            torch.save(checkpoint, temporary)\n            temporary.replace(output / "best.pt")\n        if score > monitor_best + config["min_delta"]:\n            monitor_best, stale = score, 0\n        else:\n            stale += 1\n        print(f"{kind} epoch {epoch}: train loss {history[-1][\'training_loss\']:.4f}; val loss {validation_loss:.4f}; val macro F1 {score:.4f}", flush=True)\n        if stale >= config["patience"]:\n            break\n    training_seconds = time.perf_counter() - start\n    del optimizer, model\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    model = load_checkpoint(kind, output, data, config, device)\n    probabilities, positions, loss = predict_loader(model, kind, validation, device, weights)\n    metrics = save_evaluation(output / "validation", probabilities, positions, data, loss, kind, "validation")\n    if not math.isclose(metrics["macro_f1"], best_f1, rel_tol=0, abs_tol=1e-12):\n        raise RuntimeError("Reloaded checkpoint does not reproduce validation selection score")\n    benchmark = benchmark_forward(model, kind, data, config, device)\n    write_json(output / "inference_benchmark.json", benchmark)\n    result = {"model": kind, "signature": signature, "verification": verification, "status": "complete",\n              "best_epoch": best_epoch, "epochs_run": len(history), "training_seconds_including_initialization": training_seconds,\n              "validation_macro_f1": metrics["macro_f1"], "validation_accuracy": metrics["accuracy"],\n              "checkpoint_bytes": (output / "best.pt").stat().st_size, "checkpoint_sha256": sha256(output / "best.pt"),\n              "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),\n              "device": str(device), "peak_allocated_gpu_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,\n              "batch_size": loader.batch_size, "gradient_accumulation_steps": accumulation,\n              "amp": use_amp, "preparation_id": data.manifest["preparation_id"]}\n    result["inference_batch1_median_ms"] = benchmark["median_ms"]\n    write_json(completed, result)\n    del model\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n'
target = PROJECT_DIR / 'src/complaints/training.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/training.py


In [14]:
# Embedded file: src/complaints/verify_preparation.py
embedded_source = '"""Independently verify hashes, split isolation, training-only vocabulary and IDs."""\nfrom collections import Counter\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nfrom transformers import AutoTokenizer\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(ROOT / "src"))\nfrom complaints.data import PreparedData, write_json\nfrom complaints.text import clean_recurrent, encode_recurrent\n\n\ndef verify(root):\n    data = PreparedData(root)\n    stopwords = set(data.preprocessing["stopwords"])\n    counts = Counter(word for text in data.records.loc[data.records.split.eq("train"), "narrative"]\n                     for word in clean_recurrent(text, stopwords).split())\n    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:data.config["vocabulary_cap"] - 2]\n    expected = {"<PAD>": 0, "<UNK>": 1, **{word: i + 2 for i, (word, _) in enumerate(ordered)}}\n    assert data.vocabulary == expected, "Vocabulary is not exactly the training-only vocabulary"\n    tokenizer = AutoTokenizer.from_pretrained(data.root / "tokenizer", local_files_only=True)\n    samples = np.random.default_rng(42).choice(len(data.records), size=min(256, len(data.records)), replace=False)\n    for index in samples:\n        narrative = data.records.iloc[index].narrative\n        ids = encode_recurrent(narrative, data.vocabulary, stopwords, data.config["recurrent_max_length"])\n        assert len(ids) == data.encoded["recurrent_lengths"][index]\n        np.testing.assert_array_equal(ids, data.encoded["recurrent_ids"][index, :len(ids)])\n        encoded = tokenizer(narrative, truncation=True, max_length=data.config["transformer_max_length"], padding="max_length")\n        np.testing.assert_array_equal(encoded["input_ids"], data.encoded["transformer_ids"][index])\n    for kind in ["recurrent", "transformer"]:\n        keys = [hashlib.sha256(row.astype("<i4").tobytes()).hexdigest() for row in data.encoded[kind + "_ids"]]\n        assert keys == data.records[kind + "_input_sha256"].tolist()\n        groups = data.records.groupby(kind + "_input_sha256").split.nunique()\n        assert int(groups.max()) == 1\n    summary = {"status": "passed", "preparation_id": data.manifest["preparation_id"], "records": len(data.records),\n               "checks": ["Every prepared file checksum", "Array shapes, padding and label mapping", "All five classes in every split",\n                          "No source/recurrent/truncated-input overlap between splits", "Vocabulary exactly reproduced from training only",\n                          "Class weights exactly reproduced from training only", "256 sampled records independently re-tokenized",\n                          "All encoded-input fingerprints independently recomputed"],\n               "sampled_records_reencoded": len(samples)}\n    return summary\n\n\nif __name__ == "__main__":\n    result = verify(ROOT / "prepared_data")\n    write_json(ROOT / "reports/preparation_verification.json", result)\n    print(json.dumps(result, indent=2))\n'
target = PROJECT_DIR / 'src/complaints/verify_preparation.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/verify_preparation.py


In [15]:
# Embedded file: src/complaints/worker.py
embedded_source = '"""A fresh worker sees exactly one assigned GPU before importing PyTorch."""\nimport argparse\nimport os\nfrom pathlib import Path\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--model\', required=True, choices=[\'rnn\', \'lstm\', \'gru\', \'distilbert\'])\n    parser.add_argument(\'--data\', required=True)\n    parser.add_argument(\'--config\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    args = parser.parse_args()\n    import torch\n    from .data import PreparedData, read_json, write_json\n    from .training import run_model, environment_record\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:\n        raise RuntimeError(\'Worker requires exactly one CUDA-visible GPU. Check accelerator and scheduler setup.\')\n    torch.set_num_threads(2)\n    torch.cuda.set_device(0)\n    print(f"Worker {args.model}: assigned GPU {os.environ.get(\'COMPLAINTS_ASSIGNED_GPU\')}; "\n          f"visible device cuda:0 = {torch.cuda.get_device_name(0)}", flush=True)\n    data = PreparedData(args.data, verify=True)\n    config = read_json(args.config)\n    result = run_model(args.model, data, config, args.output, torch.device(\'cuda:0\'))\n    write_json(Path(args.output) / \'models\' / args.model / \'worker_environment.json\', {\n        \'assigned_gpu\': os.environ.get(\'COMPLAINTS_ASSIGNED_GPU\'),\n        \'cuda_visible_devices\': os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n        \'runtime\': environment_record(), \'completed_signature\': result[\'signature\']})\n\n\nif __name__ == \'__main__\':\n    main()\n'
target = PROJECT_DIR / 'src/complaints/worker.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote src/complaints/worker.py


In [16]:
# Embedded file: app.py
embedded_source = '"""Gradio application for the frozen, validation-selected complaint classifier."""\nimport os\nfrom pathlib import Path\nimport sys\n\n# Text-only, offline inference. Set these before importing Transformers.\nos.environ.setdefault("USE_TF", "0")\nos.environ.setdefault("USE_FLAX", "0")\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\nos.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")\n\nimport gradio as gr\nimport torch\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT / "src"))\nfrom complaints.inference import ComplaintPredictor\nfrom complaints.text import normalize_text\n\nDISPLAY_NAMES = {\n    "credit_card": "Credit card",\n    "credit_reporting": "Credit reporting",\n    "debt_collection": "Debt collection",\n    "mortgages_and_loans": "Mortgages and loans",\n    "retail_banking": "Retail banking",\n}\nEXAMPLES = [\n    ["My credit report shows an account that I never opened. I disputed it but it is still listed."],\n    ["A debt collector keeps calling about a debt that does not belong to me."],\n    ["My mortgage servicer applied my payment incorrectly and charged a late fee."],\n    ["My credit card was charged twice and the card provider has not resolved my dispute."],\n    ["My bank account was frozen and I cannot access my deposited funds."],\n]\nCSS = """\n.gradio-container { max-width: 1120px !important; margin: auto !important; }\n#hero { padding: 18px 0 10px; }\n#hero h1 { font-size: clamp(26px, 3.8vw, 38px); font-weight: 700; letter-spacing: -0.9px; margin-bottom: 6px; }\n#hero p { font-size: 15px; color: var(--body-text-color-subdued); max-width: 820px; line-height: 1.6; }\n\n#prediction-card {\n    border: 1px solid var(--border-color-primary);\n    border-radius: 16px;\n    padding: 18px;\n    background: var(--background-fill-secondary);\n}\n\n#headline h3 {\n    font-size: 19px;\n    font-weight: 700;\n    margin-bottom: 4px;\n}\n\n/* Eliminate the distracting dotted lines */\n.confidence-set .line {\n    display: none !important;\n}\n\n/* Remove redundant giant header inside Label component */\n.output-class {\n    display: none !important;\n}\n\n/* Modern typography and card spacing for each category */\n.confidence-set {\n    font-family: inherit !important;\n    background: transparent !important;\n    border: none !important;\n    padding: 0 !important;\n    margin-bottom: 14px !important;\n    width: 100% !important;\n}\n\n.confidence-set .inner-wrap {\n    display: flex !important;\n    flex-direction: column !important;\n    width: 100% !important;\n}\n\n/* Category label on left, clean bold percentage on right */\n.confidence-set .label {\n    order: 1 !important;\n    display: flex !important;\n    justify-content: space-between !important;\n    align-items: center !important;\n    width: 100% !important;\n    margin-bottom: 6px !important;\n}\n\n.confidence-set .text {\n    font-size: 14px !important;\n    font-weight: 500 !important;\n    color: var(--body-text-color) !important;\n}\n\n.confidence-set .confidence {\n    font-size: 14px !important;\n    font-weight: 700 !important;\n    color: var(--primary-600, #2563eb) !important;\n    margin-left: auto !important;\n}\n\n/* Sleek, modern progress bar placed underneath the text */\n.confidence-set .bar {\n    order: 2 !important;\n    height: 7px !important;\n    border-radius: 6px !important;\n    background: linear-gradient(90deg, #2563eb, #06b6d4) !important;\n    margin: 0 !important;\n    align-self: flex-start !important;\n}\n\n#footer { font-size: 13px; opacity: 0.8; padding-top: 14px; }\n"""\n\n\ndef create_app(model_dir=None):\n    model_dir = Path(model_dir or os.environ.get("MODEL_DIR", ROOT / "deployment_model"))\n    if not (model_dir / "bundle.json").exists():\n        raise FileNotFoundError("The trained deployment_model bundle is missing.")\n    torch.set_num_threads(max(1, min(4, int(os.environ.get("APP_CPU_THREADS", "2")))))\n    predictor = ComplaintPredictor(model_dir, device="cpu")\n\n    def classify(text):\n        try:\n            scores = predictor.predict(text)\n        except ValueError as exc:\n            # Clear the previous prediction on invalid submissions.\n            return "### Enter a valid complaint", {}, str(exc)\n        winner = max(scores, key=scores.get)\n        input_length, cap = predictor.input_length(text)\n        status = "Prediction complete."\n        if input_length > cap:\n            status += f" This complaint exceeds {cap} tokens; the model used its opening portion."\n        headline = f"### {DISPLAY_NAMES[winner]}\\nModel probability: **{scores[winner]:.1%}**"\n        return headline, {DISPLAY_NAMES[k]: v for k, v in scores.items()}, status\n\n    model_name = {"distilbert": "DistilBERT", "rnn": "SimpleRNN", "lstm": "LSTM", "gru": "GRU"}[predictor.kind]\n    summary = ""\n    metrics_path = ROOT / "reports/final_test" / predictor.kind / "metrics.json"\n    if metrics_path.exists():\n        import json\n        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))\n        summary = f"Recorded held-out results: **{metrics[\'accuracy\']:.2%} accuracy**, **{metrics[\'macro_f1\']:.4f} macro F1**. These describe the archived experiment.\\n\\n"\n    cap = predictor.preprocessing["transformer_max_length" if predictor.kind == "distilbert" else "recurrent_max_length"]\n    theme = gr.themes.Soft(primary_hue="blue", secondary_hue="teal", neutral_hue="slate")\n    with gr.Blocks(title="Consumer Complaint Classifier", theme=theme, css=CSS) as app:\n        gr.Markdown(\n            "NLP TRAINING PROJECT · COMPLAINT CLASSIFICATION\\n\\n# Consumer Complaint Classifier\\n"\n            "A prototype supporting complaint categorization for financial products and services. "\n            "The system analyzes customer narratives in real-time to predict the primary resolution category.",\n            elem_id="hero",\n        )\n        with gr.Row(equal_height=False):\n            with gr.Column(scale=6):\n                complaint = gr.Textbox(\n                    lines=8, max_lines=15, label="Your complaint",\n                    placeholder="For example: My bank froze my account and I cannot access my deposited funds…",\n                    info="English text · up to 20,000 characters. Avoid entering private account details.",\n                )\n                with gr.Row():\n                    submit = gr.Button("Classify complaint", variant="primary")\n                    clear = gr.Button("Clear", variant="secondary")\n                status = gr.Textbox(label="Status", value="Ready for a complaint.", interactive=False)\n            with gr.Column(scale=5, elem_id="prediction-card"):\n                headline = gr.Markdown("### Your result will appear here")\n                output = gr.Label(label="Probability across all categories", num_top_classes=5, show_heading=False)\n        gr.Examples(examples=EXAMPLES, inputs=complaint, label="Try an example")\n        with gr.Accordion("Categories and model limitations", open=False):\n            gr.Markdown(\n                "**Categories:** credit cards, credit reporting, debt collection, mortgages and loans, and retail banking.\\n\\n"\n                f"Loaded model: **{model_name}**.\\n\\n"\n                + summary +\n                "Scores are uncalibrated model probabilities. The model always chooses among the five categories, "\n                "including for unrelated text. It was trained on narratives already processed by the publisher, "\n                f"so naturally written complaints may behave differently. Long complaints are truncated to {cap} tokens."\n            )\n        gr.Markdown("Consumer Complaint Classification · Training project prototype", elem_id="footer")\n        outputs = [headline, output, status]\n        submit.click(classify, inputs=complaint, outputs=outputs, api_name="classify", concurrency_limit=1)\n        complaint.submit(classify, inputs=complaint, outputs=outputs, api_name=False, concurrency_limit=1)\n        clear.click(lambda: ("", "### Your result will appear here", {}, "Ready for a complaint."),\n                    outputs=[complaint, headline, output, status], api_name=False, queue=False)\n    return app.queue(default_concurrency_limit=1, max_size=24)\n\n\nif __name__ == "__main__":\n    port = int(os.environ.get("PORT", os.environ.get("GRADIO_SERVER_PORT", "7860")))\n    server_name = os.environ.get("SERVER_NAME", "0.0.0.0" if os.environ.get("SPACE_ID") else "127.0.0.1")\n    create_app().launch(server_name=server_name, server_port=port,\n                        show_error=False, share=False)\n'
target = PROJECT_DIR / 'app.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote app.py


In [17]:
# Embedded file: requirements.txt
embedded_source = '# Application dependencies. Training uses requirements-training.txt.\ntorch==2.5.1\ntransformers==4.57.1\ntokenizers==0.22.2\nhuggingface-hub==0.36.2\nsafetensors==0.8.0\nnumpy==2.3.5\ngradio==5.49.1\nhttpx[socks]>=0.24,<1\n'
target = PROJECT_DIR / 'requirements.txt'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote requirements.txt


In [18]:
# Embedded file: configs/experiment.json
embedded_source = '{\n  "schema_version": 1,\n  "seed": 42,\n  "models": [\n    "rnn",\n    "lstm",\n    "gru",\n    "distilbert"\n  ],\n  "selection_metric": "validation_macro_f1",\n  "tie_tolerance": 0.001,\n  "tie_priority": [\n    "rnn",\n    "gru",\n    "lstm",\n    "distilbert"\n  ],\n  "recurrent_max_length": 256,\n  "transformer_max_length": 256,\n  "vocabulary_cap": 30000,\n  "embedding_dim": 128,\n  "hidden_dim": 128,\n  "dropout": 0.3,\n  "recurrent_batch_size": 128,\n  "transformer_batch_size": 16,\n  "eval_batch_size": 64,\n  "gradient_accumulation_steps": 2,\n  "recurrent_epochs": 8,\n  "transformer_epochs": 3,\n  "patience": 2,\n  "min_delta": 0.0001,\n  "recurrent_lr": 0.001,\n  "transformer_lr": 2e-05,\n  "transformer_weight_decay": 0.01,\n  "warmup_ratio": 0.1,\n  "max_grad_norm": 1.0,\n  "mixed_precision": true,\n  "num_workers": 0,\n  "class_weight_policy": "balanced_training_only",\n  "transformer_checkpoint": "distilbert/distilbert-base-uncased",\n  "transformer_revision": "12040accade4e8a0f71eabdb258fecc2e7e948be",\n  "truncation_policy": "head",\n  "final_test": true,\n  "resume_interrupted": false,\n  "benchmark_repeats": 30\n}\n'
target = PROJECT_DIR / 'configs/experiment.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote configs/experiment.json


In [19]:
# Embedded file: configs/reference.json
embedded_source = '{\n  "audit": {\n    "source": "shashwatwork/consume-complaints-dataset-fo-nlp",\n    "source_version": 1,\n    "retrieved_date": "2026-11-07",\n    "source_archive_sha256": "45e406e64bba3a77dc76a66b096eb54c7b5e5a896be4dd851503e505465c6599",\n    "source_csv_sha256": "a120ac8450cb3839bf0085bbae68f7c0f3b5351302cafd0bc85ae27a251010b4",\n    "source_rows": 162421,\n    "source_columns": [\n      "Unnamed: 0",\n      "product",\n      "narrative"\n    ],\n    "source_class_counts": {\n      "credit_reporting": 91179,\n      "debt_collection": 23150,\n      "mortgages_and_loans": 18990,\n      "credit_card": 15566,\n      "retail_banking": 13536\n    },\n    "official_splits": "None: one CSV with no split column",\n    "normalized_conflicting_groups": 172,\n    "recurrent_conflicting_groups_after_source_dedup": 10,\n    "exclusion_flow": [\n      {\n        "stage": "source",\n        "remaining": 162421,\n        "excluded": 0\n      },\n      {\n        "stage": "blank_text_or_label",\n        "remaining": 162411,\n        "excluded": 10\n      },\n      {\n        "stage": "conflicting_source_text_labels",\n        "remaining": 159134,\n        "excluded": 3277\n      },\n      {\n        "stage": "duplicate_source_text",\n        "remaining": 124300,\n        "excluded": 34834\n      },\n      {\n        "stage": "empty_after_recurrent_cleaning",\n        "remaining": 124300,\n        "excluded": 0\n      },\n      {\n        "stage": "conflicting_recurrent_text_labels",\n        "remaining": 124278,\n        "excluded": 22\n      },\n      {\n        "stage": "duplicate_recurrent_text",\n        "remaining": 123978,\n        "excluded": 300\n      }\n    ],\n    "retained_rows": 123978,\n    "labels": [\n      "credit_card",\n      "credit_reporting",\n      "debt_collection",\n      "mortgages_and_loans",\n      "retail_banking"\n    ],\n    "split_counts": {\n      "train": 99182,\n      "validation": 12398,\n      "test": 12398\n    },\n    "class_counts_by_split": {\n      "test": {\n        "credit_card": 1496,\n        "credit_reporting": 5589,\n        "debt_collection": 2097,\n        "mortgages_and_loans": 1870,\n        "retail_banking": 1346\n      },\n      "train": {\n        "credit_card": 11966,\n        "credit_reporting": 44712,\n        "debt_collection": 16777,\n        "mortgages_and_loans": 14959,\n        "retail_banking": 10768\n      },\n      "validation": {\n        "credit_card": 1496,\n        "credit_reporting": 5590,\n        "debt_collection": 2097,\n        "mortgages_and_loans": 1869,\n        "retail_banking": 1346\n      }\n    },\n    "lengths": {\n      "train": {\n        "records": 99182,\n        "recurrent_length_percentiles": {\n          "50": 51.0,\n          "90": 158.0,\n          "95": 222.0,\n          "99": 450.0,\n          "100": 2432.0\n        },\n        "transformer_length_percentiles": {\n          "50": 66.0,\n          "90": 202.0,\n          "95": 282.0,\n          "99": 572.0,\n          "100": 3312.0\n        },\n        "recurrent_truncation_fraction": 0.03660946542719445,\n        "transformer_truncation_fraction": 0.06084773446794781,\n        "recurrent_unknown_token_fraction": 0.0015161892173136118\n      },\n      "validation": {\n        "records": 12398,\n        "recurrent_length_percentiles": {\n          "50": 50.0,\n          "90": 154.0,\n          "95": 213.0,\n          "99": 399.0,\n          "100": 2344.0\n        },\n        "transformer_length_percentiles": {\n          "50": 66.0,\n          "90": 198.0,\n          "95": 272.0,\n          "99": 495.1200000000026,\n          "100": 3054.0\n        },\n        "recurrent_truncation_fraction": 0.03355379899983869,\n        "transformer_truncation_fraction": 0.056864010324245845,\n        "recurrent_unknown_token_fraction": 0.0036775013931692243\n      },\n      "test": {\n        "records": 12398,\n        "recurrent_length_percentiles": {\n          "50": 51.0,\n          "90": 160.0,\n          "95": 226.0,\n          "99": 434.1200000000026,\n          "100": 1826.0\n        },\n        "transformer_length_percentiles": {\n          "50": 66.0,\n          "90": 206.0,\n          "95": 286.0,\n          "99": 550.0,\n          "100": 2175.0\n        },\n        "recurrent_truncation_fraction": 0.037102758509437006,\n        "transformer_truncation_fraction": 0.06202613324729795,\n        "recurrent_unknown_token_fraction": 0.003390192770317282\n      }\n    },\n    "vocabulary_size_including_specials": 30000,\n    "training_distinct_words": 41519,\n    "training_class_weights": {\n      "credit_card": 1.6577302356677253,\n      "credit_reporting": 0.44364823760959027,\n      "debt_collection": 1.1823567979972582,\n      "mortgages_and_loans": 1.3260512066314594,\n      "retail_banking": 1.8421619613670133\n    },\n    "duplicate_overlap_between_splits": {\n      "normalized_source_text": 0,\n      "recurrent_cleaned_text": 0,\n      "recurrent_encoded_input": 0,\n      "transformer_encoded_input": 0\n    },\n    "split_policy": "10 StratifiedGroupKFold folds with seed 42: fold 0 validation, fold 1 test, other folds train. Truncated-input equivalence groups stay together.",\n    "split_group_count": 123761,\n    "multirow_split_groups": 123,\n    "encoded_collision_grouping_passes": 3,\n    "source_preprocessing_evidence": {\n      "uppercase_narratives": 0,\n      "digit_narratives": 0,\n      "non_lowercase_letter_or_whitespace_narratives": 0,\n      "raw_narratives_available": false\n    },\n    "near_duplicate_policy": "Exact normalized and recurrent-cleaned duplicates removed. Semantic/paraphrase near-duplicates are not guaranteed absent.",\n    "upstream_limitations": "Unknown upstream cleaning/label consolidation; no raw text or original complaint identifiers. No original-data linkage is invented.",\n    "training_status": "NOT TRAINED",\n    "preparation_versions": {\n      "numpy": "2.3.5",\n      "pandas": "2.2.3",\n      "scikit-learn": "1.8.0",\n      "transformers": "4.57.1",\n      "tokenizers": "0.22.2"\n    }\n  },\n  "records_csv_sha256": "1a53c0ca7ee3714a598cbd0e88f4e71ce041ac3b9beb49c8230948efaee5ef83",\n  "vocabulary_sha256": "7ecf093b670a1a18a1039ba113de8055391c8f263207d882ce0042be9255347e",\n  "npy_sha256": {\n    "recurrent_ids.npy": "15db89b50a21bac7480451a1528cd7cca1ad1bab980384596305e0c99072a032",\n    "recurrent_lengths.npy": "52daa32be4c3afbff332e6dd7dd91adfe817e7a505a0c9bd73b8dba779c72b59",\n    "transformer_ids.npy": "486930d211ea8d02e1d124986526589291516185f5ab3e9cf02554f8530ed858",\n    "transformer_lengths.npy": "f9a6bb1a409b2777d58c578fdd83e33b1c9c35cdfca6a55a938a8db771d246d7",\n    "recurrent_full_lengths.npy": "7fc47872eb6bfea9afc2c8b828dd6b7c85c0da3fa7113b285f3da58d7ff76a11",\n    "transformer_full_lengths.npy": "32e9f4fa3626e3c1a87d20e69636ba30448628eb5ff543302550920be4eb5cf8"\n  }\n}'
target = PROJECT_DIR / 'configs/reference.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(embedded_source, encoding="utf-8")
print("Wrote", target.relative_to(PROJECT_DIR))


Wrote configs/reference.json


## Run configuration

Defaults run bounded verification, then all four full experiments, validation selection, the final test phase, and app export. Batch sizes can be reduced before starting if a different GPU has less memory. Model input lengths are fixed by the prepared data.

Use a new `OUTPUT_DIR` when changing experiment settings. Compatible completed models are reused automatically. An interrupted model restarts from epoch 1; weights-only files do not resume optimizer state. `MODELS_TO_RUN` can limit a session to selected architectures; final comparison/export waits until all four compatible full runs exist.

Defaults use up to two visible GPUs. `MAX_GPUS = 1` or `USE_PARALLEL_GPUS = False` selects sequential GPU execution without changing model hyperparameters. Keep the default full run for all four models.


## Download, prepare and verify against the reference

KaggleHub fetches `shashwatwork/consume-complaints-dataset-fo-nlp/versions/1`. In Kaggle it can mount shared input storage automatically; generated files and caches are written under `/kaggle/working/`. No manual input upload is needed. KaggleHub is normally authenticated by the Kaggle notebook session; no credential is embedded. Saved outputs record the historical source download; reruns verify the pinned source checksum. If access fails, inspect Internet/account/consent settings and rerun; no alternate dataset is substituted.

Preparation is cached only with matching verified source, configuration, implementation, dependency versions and reference fingerprints. A changed source or incompatible/corrupt preparation stops before training. Reference checks include record order and split assignments, vocabulary, all six encoded arrays, class counts and exclusions. Differences produce a diagnostic report rather than quietly changing the experiment.


In [20]:
# Source acquisition and complete reproducible preparation; executed in Kaggle.
sys.path.insert(0, str(PROJECT_DIR / "src"))
from complaints.bootstrap import prepare_all
PREPARED_DIR = prepare_all(PROJECT_DIR, Path("/kaggle/working/consumer_complaints_prepared"))
print("Prepared and reference-verified:", PREPARED_DIR)


Acquiring source version 1 directly inside Kaggle...
Tokenized 1,024/123,978
Tokenized 21,504/123,978
Tokenized 41,984/123,978
Tokenized 62,464/123,978
Tokenized 82,944/123,978
Tokenized 103,424/123,978
Tokenized 123,904/123,978
Split verification pass 1: 1 recurrent input collisions across splits
Split verification pass 2: 2 recurrent input collisions across splits
Split verification pass 3: 0 recurrent input collisions across splits
{
  "retained": 123978,
  "splits": {
    "train": 99182,
    "validation": 12398,
    "test": 12398
  },
  "preparation_id": "4e3ead3e2e205566cb08dfb02fb7138ccc924c580f462808e3db5bc665f19b04",
  "lengths": {
    "train": {
      "records": 99182,
      "recurrent_length_percentiles": {
        "50": 51.0,
        "90": 158.0,
        "95": 222.0,
        "99": 450.0,
        "100": 2432.0
      },
      "transformer_length_percentiles": {
        "50": 66.0,
        "90": 202.0,
        "95": 282.0,
        "99": 572.0,
        "100": 3312.0
      },
   

In [21]:
import gc
sys.path.insert(0, str(PROJECT_DIR / "src"))
from complaints.data import PreparedData, find_prepared, read_json, write_json
from complaints.training import environment_record, run_model, experiment_signature
from complaints.pipeline import REQUIRED_MODELS, run_verification, freeze_selection, final_test, export_selected

RUN_MODE = os.environ.get("COMPLAINTS_RUN_MODE", "full")  # full, smoke, or verify_only
RUN_FUNCTIONAL_CHECKS = True
USE_PARALLEL_GPUS = True  # one independent model per GPU
MAX_GPUS = 2
MODELS_TO_RUN = ["rnn", "lstm", "gru", "distilbert"]
DATA_DIR = None  # generated by the preceding preparation stage
OUTPUT_DIR = Path(os.environ.get("COMPLAINTS_OUTPUT_DIR", "/kaggle/working/consumer_complaints_run"))
DEVICE = os.environ.get("COMPLAINTS_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
if RUN_MODE not in {"full", "smoke", "verify_only"}:
    raise ValueError("RUN_MODE must be full, smoke, or verify_only")
if RUN_MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before a full training run.")
if not MODELS_TO_RUN or len(set(MODELS_TO_RUN)) != len(MODELS_TO_RUN) or not set(MODELS_TO_RUN).issubset(REQUIRED_MODELS):
    raise ValueError("Unknown architecture in MODELS_TO_RUN")
torch.set_num_threads(min(2, os.cpu_count() or 1))
visible_count = torch.cuda.device_count()
if visible_count:
    inherited_visibility = os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",")
    physical_ids = ([x.strip() for x in inherited_visibility if x.strip()]
                    if len([x for x in inherited_visibility if x.strip()]) == visible_count
                    else [str(i) for i in range(visible_count)])
    GPU_IDS = physical_ids[:MAX_GPUS] if USE_PARALLEL_GPUS else physical_ids[:1]
else:
    GPU_IDS = []
print("Worker GPU assignments available:", GPU_IDS)
if len(GPU_IDS) == 1:
    print("One GPU visible: models will run sequentially. Select T4 x2 to overlap training.")
data_root = Path(DATA_DIR) if DATA_DIR else PREPARED_DIR
data = PreparedData(data_root, verify=True)
config = dict(data.config)
# Optional adjustments BEFORE the full run; record all changes with a new OUTPUT_DIR.
# config["transformer_batch_size"] = 8
# config["gradient_accumulation_steps"] = 4
assert config["recurrent_max_length"] == data.preprocessing["recurrent_max_length"]
assert config["transformer_max_length"] == data.preprocessing["transformer_max_length"]
if RUN_MODE == "full" and not config["final_test"]:
    raise RuntimeError("The full deliverable includes the final held-out test phase; keep final_test true.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import shutil
prep_reports = OUTPUT_DIR / "preparation_reports"
prep_reports.mkdir(exist_ok=True)
for name in ["audit.json", "reference_comparison.json", "preparation_complete.json", "source_row_ledger.csv.gz", "labels.json"]:
    shutil.copy2(PREPARED_DIR / name, prep_reports / name)
for name in ["source_download.json", "class_counts_by_split.csv"]:
    origin = PROJECT_DIR / "reports" / name
    if origin.exists(): shutil.copy2(origin, prep_reports / name)
shutil.copy2(PROJECT_DIR / "configs/reference.json", prep_reports / "reference_fingerprints.json")
figure = PROJECT_DIR / "reports/figures/data_overview.png"
if figure.exists(): shutil.copy2(figure, prep_reports / figure.name)
signature, identity = experiment_signature(data, config)
write_json(OUTPUT_DIR / "runtime_environment.json", environment_record())
print("Prepared input:", data_root)
print("Records:", len(data.records), "Classes:", data.label_names)
print("Device:", DEVICE, "Mode:", RUN_MODE)
print("Experiment signature:", signature)
print("Split sizes:", data.records.split.value_counts().to_dict())
display(data.records.groupby(["product", "split"]).size().unstack())


Worker GPU assignments available: ['0', '1']
Prepared input: /kaggle/working/consumer_complaints_prepared
Records: 123978 Classes: ['credit_card', 'credit_reporting', 'debt_collection', 'mortgages_and_loans', 'retail_banking']
Device: cuda:0 Mode: full
Experiment signature: bd457ca03f08e8a79677243e610ca2393dc21aefe70fde9dc10db0f32b83c23a
Split sizes: {'train': 99182, 'validation': 12398, 'test': 12398}


split,test,train,validation
product,,,
credit_card,1496,11966,1496
credit_reporting,5589,44712,5590
debt_collection,2097,16777,2097
mortgages_and_loans,1870,14959,1869
retail_banking,1346,10768,1346


## Data audit and readiness checks

Prepared inputs are verified by checksum, shape, label identity, class weights, and split isolation, including duplicates introduced by truncation and unknown tokens. The audit below records all exclusions. Source text was processed upstream; no unobserved original text is reconstructed.


In [22]:
audit = read_json(data.root / "audit.json")
display(__import__("pandas").DataFrame(audit["exclusion_flow"]))
display(__import__("pandas").DataFrame(audit["lengths"]).T)
print("No identical source/recurrent/encoded inputs cross splits:", audit["duplicate_overlap_between_splits"])
print("Vocabulary size:", len(data.vocabulary))
print("Training-derived class weights:", audit["training_class_weights"])
print("Important limitation:", audit["upstream_limitations"])


,stage,remaining,excluded
0,source,162421,0
1,blank_text_or_label,162411,10
2,conflicting_source_text_labels,159134,3277
3,duplicate_source_text,124300,34834
4,empty_after_recurrent_cleaning,124300,0
5,conflicting_recurrent_text_labels,124278,22
6,duplicate_recurrent_text,123978,300


,records,recurrent_length_percentiles,transformer_length_percentiles,recurrent_truncation_fraction,transformer_truncation_fraction,recurrent_unknown_token_fraction
train,99182,"{'50': 51.0, '90': 158.0, '95': 222.0, '99': 4...","{'50': 66.0, '90': 202.0, '95': 282.0, '99': 5...",0.036609,0.060848,0.001516
validation,12398,"{'50': 50.0, '90': 154.0, '95': 213.0, '99': 3...","{'50': 66.0, '90': 198.0, '95': 272.0, '99': 4...",0.033554,0.056864,0.003678
test,12398,"{'50': 51.0, '90': 160.0, '95': 226.0, '99': 4...","{'50': 66.0, '90': 206.0, '95': 286.0, '99': 5...",0.037103,0.062026,0.00339


No identical source/recurrent/encoded inputs cross splits: {'normalized_source_text': 0, 'recurrent_cleaned_text': 0, 'recurrent_encoded_input': 0, 'transformer_encoded_input': 0}
Vocabulary size: 30000
Training-derived class weights: {'credit_card': 1.6577302356677253, 'credit_reporting': 0.44364823760959027, 'debt_collection': 1.1823567979972582, 'mortgages_and_loans': 1.3260512066314594, 'retail_banking': 1.8421619613670133}
Important limitation: Unknown upstream cleaning/label consolidation; no raw text or original complaint identifiers. No original-data linkage is invented.


## Bounded functional verification

Each architecture trains on **one training example per class for one epoch**, evaluates a tiny validation sample, saves/reloads its checkpoint, and checks CPU-export consistency. These outputs stay under `verification_only/` and cannot enter full model selection. No test records are evaluated in this stage.

`RUN_MODE="smoke"` runs full data preparation and tiny functional checks but skips full training, selection, test and final exports. `verify_only` checks preparation without neural training. Full mode is the default. Smoke metrics are not experiment results.


In [23]:
if RUN_MODE != "verify_only" and RUN_FUNCTIONAL_CHECKS:
    verification = run_verification(data, config, OUTPUT_DIR, DEVICE, kinds=MODELS_TO_RUN)
    print("Functional checks passed for:", list(verification))
else:
    print("Training verification skipped in verify_only mode.")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


rnn epoch 1: train loss 1.5361; val loss 1.4439; val macro F1 0.7333
lstm epoch 1: train loss 1.6294; val loss 1.6515; val macro F1 0.2333
gru epoch 1: train loss 1.6562; val loss 1.5528; val macro F1 0.3333


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


distilbert epoch 1: train loss 1.6844; val loss 1.6088; val macro F1 0.1000
Functional checks passed for: ['rnn', 'lstm', 'gru', 'distilbert']


## Full training — independent models on both GPUs

The queue starts DistilBERT first and SimpleRNN alongside it. Each worker sees only its assigned GPU and uses local `cuda:0`; worker logs explicitly record the physical assignment. Seeing `cuda:0` inside both logs is expected. At most one model occupies each GPU. Each finished worker releases its GPU for the next model.

Batch sizes remain 128 for recurrent models and 16 with two-step accumulation for DistilBERT (effective batch 32). Mixed precision, class weighting, clipping and early stopping remain enabled. The transformer is not duplicated across GPUs. Validation for one model can overlap training for another; each individual experiment's duration reflects this concurrent workload.

Live logs are shown below and saved under `worker_logs/`. A worker error stops remaining workers, retains completed checkpoints, and blocks selection/export. Correct the cause before rerunning. An interrupted architecture restarts at epoch 1; completed compatible models are reused. Final test evaluation and export run after training completes.


In [24]:
from complaints.parallel import train_models_parallel
training_results = {}
if RUN_MODE == "full":
    training_results = train_models_parallel(data, config, OUTPUT_DIR, MODELS_TO_RUN, GPU_IDS, PROJECT_DIR)
    display(__import__("pandas").DataFrame(training_results.values())[
        ["model", "validation_macro_f1", "validation_accuracy", "best_epoch", "epochs_run", "training_seconds_including_initialization"]])
    display(read_json(OUTPUT_DIR / "parallel_execution.json"))
else:
    print("Full training skipped:", RUN_MODE)


Started distilbert on GPU 0; log: /kaggle/working/consumer_complaints_run/worker_logs/distilbert.log
Started rnn on GPU 1; log: /kaggle/working/consumer_complaints_run/worker_logs/rnn.log
[distilbert / GPU 0]
Worker distilbert: assigned GPU 0; visible device cuda:0 = Tesla T4
[rnn / GPU 1]
Worker rnn: assigned GPU 1; visible device cuda:0 = Tesla T4
[rnn / GPU 1]

rnn pilot: 0.020 s/batch; about 2.1 min for 8 training epochs, excluding validation/downloads.
[distilbert / GPU 0]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[distilbert / GPU 0]

distilbert pilot: 0.133 s/batch; about 41.2 min for 3 training epochs, excluding validation/downloads.
Running: distilbert on GPU 0

,model,validation_macro_f1,validation_accuracy,best_epoch,epochs_run,training_seconds_including_initialization
0,rnn,0.731799,0.750847,3,5,80.982707
1,lstm,0.829397,0.842233,7,8,131.995450
2,gru,0.831587,0.844814,7,8,131.292309
3,distilbert,0.842768,0.850460,3,3,2626.894990


{'strategy': 'independent models; one process per assigned GPU',
 'gpu_ids': ['0', '1'],
 'assignments': [{'model': 'distilbert',
   'assigned_gpu': '0',
   'started_seconds': 0.0006461159999844313,
   'log': '/kaggle/working/consumer_complaints_run/worker_logs/distilbert.log',
   'elapsed_seconds': 2762.957382632,
   'status': 'complete'},
  {'model': 'rnn',
   'assigned_gpu': '1',
   'started_seconds': 0.003117257999974754,
   'log': '/kaggle/working/consumer_complaints_run/worker_logs/rnn.log',
   'elapsed_seconds': 102.04931213299994,
   'status': 'complete'},
  {'model': 'lstm',
   'assigned_gpu': '1',
   'started_seconds': 103.05435761399985,
   'log': '/kaggle/working/consumer_complaints_run/worker_logs/lstm.log',
   'elapsed_seconds': 150.0674490030001,
   'status': 'complete'},
  {'model': 'gru',
   'assigned_gpu': '1',
   'started_seconds': 254.12384463800004,
   'log': '/kaggle/working/consumer_complaints_run/worker_logs/gru.log',
   'elapsed_seconds': 150.06898773900002,
  

## Freeze validation selection before opening the test set

The highest validation macro F1 wins. For models within **0.001 absolute F1** of the best, the predeclared smaller-model priority is SimpleRNN → GRU → LSTM → DistilBERT. This rule is fixed before test evaluation. The winner cannot change in response to test results.


In [25]:
selection = None
all_full_models_ready = all((OUTPUT_DIR / "models" / name / "result.json").exists() for name in REQUIRED_MODELS)
if RUN_MODE == "full" and all_full_models_ready:
    selection = freeze_selection(data, config, OUTPUT_DIR)
    display(__import__("pandas").read_csv(OUTPUT_DIR / "validation_comparison.csv"))
elif RUN_MODE == "full":
    print("Partial training session complete. Run the remaining architectures in the same experiment before final selection.")
else:
    print("Selection skipped; verification outputs are not final experiments.")


Frozen deployment selection: distilbert


,model,validation_macro_f1,validation_accuracy,best_epoch,epochs_run,training_seconds_including_initialization,checkpoint_bytes,trainable_parameters,inference_batch1_median_ms
0,distilbert,0.842768,0.850460,3,3,2626.894990,267863151,66957317,5.888671
1,gru,0.831587,0.844814,7,8,131.292309,15761873,3939717,1.912086
2,lstm,0.829397,0.842233,7,8,131.995450,15893969,3972741,1.649692
3,rnn,0.731799,0.750847,3,5,80.982707,15497681,3873669,1.563132


## Final held-out test phase

All four frozen best checkpoints are evaluated for accuracy, macro/weighted precision, recall and F1, per-class results, raw and normalized confusion matrices, and error/truncation analysis. This is reporting only. Rerunning reuses compatible completed test outputs.


In [26]:
test_summary = None
if selection is not None:
    test_summary = final_test(data, config, OUTPUT_DIR, torch.device(DEVICE))
    display(__import__("pandas").read_csv(OUTPUT_DIR / "final_test/test_comparison.csv"))
else:
    print("Final test is locked until four full models have passed validation selection.")


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,selected_on_validation
0,rnn,0.747217,0.713404,0.748391,0.726252,0.766745,0.747217,0.751391,False
1,lstm,0.837393,0.812644,0.835777,0.823141,0.842523,0.837393,0.838855,False
2,gru,0.841426,0.818507,0.836192,0.826310,0.846342,0.841426,0.842926,False
3,distilbert,0.854331,0.831984,0.865200,0.846683,0.861484,0.854331,0.855725,True


## Export, CPU inference check and downloadable outputs

The selected model is exported with its exact tokenizer/vocabulary, labels and preprocessing. A clean reload must match evaluation probabilities. The notebook also creates a ready-to-run Gradio application archive and a results archive retaining all four best checkpoints and reports. CPU batch-one latency is measured with preprocessing included.

Use **Save & Run All** to preserve Kaggle outputs. After completion, download the results archive and Gradio app archive from the Output panel. The app archive can be deployed to a Gradio Hugging Face Space after reviewing its model results.


In [27]:
artifacts = None
if selection is not None:
    artifacts = export_selected(data, config, OUTPUT_DIR)
    print("Selected model:", selection["selected_model"])
    print("CPU benchmark:", read_json(OUTPUT_DIR / "cpu_inference_benchmark.json"))
    from IPython.display import FileLink, display
    for label, path in artifacts.items():
        print(label, path)
        display(FileLink(path))
    write_json(OUTPUT_DIR / "READY_FOR_REVIEW.json", {"status": "trained_evaluated_exported", "selection": selection,
                                                       "artifacts": artifacts, "full_training": True})
else:
    print("Verification/input checks complete. No final training metrics or trained application are claimed.")


Selected model: distilbert
CPU benchmark: {'device': 'cpu', 'batch_size': 1, 'warmup_calls': 3, 'measured_calls': 30, 'median_ms': 70.65348950004591, 'p95_ms': 126.69358914999975, 'includes_preprocessing_and_tokenization': True, 'threads': 2, 'recurrent_max_length': 256, 'transformer_max_length': 256, 'validation_reload_max_probability_difference': 4.172325134277344e-07}
reports_archive /kaggle/working/consumer_complaints_run_reports.zip


/kaggle/working/consumer_complaints_run_reports.zip

results_archive /kaggle/working/consumer_complaints_run_results.zip


/kaggle/working/consumer_complaints_run_results.zip

app_archive /kaggle/working/consumer_complaints_run/consumer_complaints_gradio_app.zip


/kaggle/working/consumer_complaints_run/consumer_complaints_gradio_app.zip

## Reruns, output retention and remaining checks

Use Save & Run All to retain outputs. Return `consumer_complaints_run_reports.zip` for analysis; the app ZIP includes selected weights and app source, and the larger results ZIP retains all four best checkpoints.

Same-session reruns verify generated preparation and reuse compatible completed models. Interrupted training restarts that architecture at epoch 1: optimizer-state resumption is not implemented. Worker failure stops other active workers, leaves completed checkpoints and blocks final selection.

New Kaggle sessions do not restore working files automatically. To reuse work, attach the previous saved notebook outputs (not a locally uploaded dataset), then copy its generated `consumer_complaints_prepared/` and `consumer_complaints_run/` directories into matching writable paths under `/kaggle/working/` before running the notebook. When only the results ZIP was saved, extract it into the experiment output directory; preparation can be regenerated deterministically. Reuse requires the same code/configuration/dependencies. A new configuration requires a new output directory.

No host publication is performed. CPU inference, confidence limitations, upstream processed text, head truncation, single-seed comparison and one-model-per-GPU timing limitations remain as documented.

The displayed outputs record the original completed Kaggle run. Later application/source maintenance was checked locally, without repeating full GPU training. Use a new output directory for a new source signature.
